In [ ]:
# ============================================================
# PROTOCOL ENGINE v2.4.4 — P18 Hybrid Model + VIX duration tracking
# ============================================================
import pandas as pd
import requests
import zipfile
import io
import os
import json
import re
from collections import Counter
from datetime import datetime, timedelta
from IPython.display import display, HTML

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG — BASIC
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FRED_KEY = "ffc67e085d555e2dca3d514470edf6b0"
SESSION_DATA_DIR = r"C:\Users\Chege\Documents\forex\forex learnings\Semi_automation"
COT_DATA_DIR = os.path.join(SESSION_DATA_DIR, "cot_data")

FOREIGN_YIELDS = {
    'EUR': {'yield': 2.56, 'updated': '2026-04-01'},
    'GBP': {'yield': 4.31, 'updated': '2026-04-01'},
    'JPY': {'yield': 1.36, 'updated': '2026-04-01'},
    'AUD': {'yield': 4.61, 'updated': '2026-04-01'},
    'NZD': {'yield': 3.46, 'updated': '2026-04-01'},
    'CAD': {'yield': 2.79, 'updated': '2026-04-01'},
    'CHF': {'yield': 0.05, 'updated': '2026-04-01'},
}

# Manual event entries (API events prepended automatically)
UPCOMING_EVENTS = [
    # {'datetime_str': '2025-07-02 14:00', 'event': 'FOMC Decision'},
]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MACRO INPUTS — Three-tier structure (CHANGE 3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CURRENT_SESSION = 'OVERLAP'   # LONDON | NY | OVERLAP | ASIA

GLOBAL_LIQUIDITY = 'EXPANSION'
# Options: EXPANSION | LEAN_EXPANSION | NEUTRAL | LEAN_CONTRACTION | CONTRACTION | CRISIS

GLOBAL_RISK_SENTIMENT = 'EXTREME_RISK_OFF'
# Options: AUTO | RISK_ON | NEUTRAL | MODERATE_RISK_OFF | SEVERE_RISK_OFF | EXTREME_RISK_OFF

FISCAL_BIAS = {
    'USD': {'direction': 'BEARISH', 'intensity': 'STRONG', 'updated': '2026-04-01'},    # Unfunded $1.9T deficit, r>g, doom loop prerequisites met
    'EUR': {'direction': 'NEUTRAL', 'intensity': None, 'updated': '2026-04-01'},        # Unanimous neutral, German funded expansion offsets France/Italy drag
    'GBP': {'direction': 'BEARISH', 'intensity': 'MILD', 'updated': '2026-04-01'},      # Neutral-to-bearish consensus, highest G7 yields, r>g
    'JPY': {'direction': 'BEARISH', 'intensity': 'MODERATE', 'updated': '2026-04-01'},  # Most fragile fiscal/monetary collision, elevated doom loop risk
    'AUD': {'direction': 'BULLISH', 'intensity': 'STRONG', 'updated': '2026-04-01'},    # Unanimous bullish, AAA rating, stable debt/GDP
    'NZD': {'direction': 'NEUTRAL', 'intensity': None, 'updated': '2026-04-01'},        # Neutral consensus, limited fiscal data impact
    'CAD': {'direction': 'NEUTRAL', 'intensity': None, 'updated': '2026-04-01'},        # Neutral consensus, political shift uncertain
    'CHF': {'direction': 'BULLISH', 'intensity': 'STRONG', 'updated': '2026-04-01'},    # Unanimous bullish, structural surplus, safe haven
}

RETAIL_SENTIMENT = {
    # GROUP A — VALIDATED can be STANDARD, HIGH OR MAXIMUM in the tier
    'EUR/USD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'GBP/USD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'USD/JPY': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'AUD/USD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'USD/CAD': {'signal': 'CONTRARIAN_BULL', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    'USD/CHF': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'NZD/USD': {'signal': 'CONTRARIAN_BEAR', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    'XAU/USD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'NAS100':  {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'SPX500':  {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'US30':    {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'WTI':     {'signal': 'CONTRARIAN_BEAR', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    # GROUP B — CONDITIONAL
    'EUR/JPY': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'GBP/JPY': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'AUD/JPY': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'NZD/JPY': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'CAD/JPY': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'CHF/JPY': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'EUR/GBP': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'EUR/AUD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'EUR/CAD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'EUR/NZD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'EUR/CHF': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'GBP/AUD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'GBP/CAD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'GBP/NZD': {'signal': 'CONTRARIAN_BULL', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    'GBP/CHF': {'signal': 'CONTRARIAN_BEAR', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    'AUD/NZD': {'signal': 'CONTRARIAN_BULL', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    'AUD/CAD': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'AUD/CHF': {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None},
    'NZD/CAD': {'signal': 'CONTRARIAN_BEAR', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    'NZD/CHF': {'signal': 'CONTRARIAN_BEAR', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
    'CAD/CHF': {'signal': 'CONTRARIAN_BEAR', 'tier': 'STANDARD', 'sources': 1, 'updated': '2026-04-30'},
}

INSTRUMENT_REGIME = {
    # 'EUR/USD': {'regime': 'RANGING', 'adx': 18.5},
}

COT_WEEKLY_ADX = {
    # 'EUR': 22.3,
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# INSTRUMENT DNA DICTIONARY (CHANGE 4) — static, never changes
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSTRUMENT_DNA = {
    'EUR/USD': {'long_char': 'MILD_RISK_ON', 'short_char': 'RISK_OFF', 'category': 'major', 'session': 'LONDON_NY', 'spread_tier': 1},
    'GBP/USD': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'major', 'session': 'LONDON_NY', 'spread_tier': 1},
    'USD/JPY': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'major', 'session': 'ALL', 'spread_tier': 1},
    'AUD/USD': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'major', 'session': 'LONDON_NY', 'spread_tier': 1},
    'NZD/USD': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'major', 'session': 'LONDON_NY', 'spread_tier': 1},
    'USD/CAD': {'long_char': 'RISK_OFF', 'short_char': 'RISK_ON', 'category': 'major', 'session': 'LONDON_NY', 'spread_tier': 1},
    'USD/CHF': {'long_char': 'NEUTRAL', 'short_char': 'RISK_OFF', 'category': 'major', 'session': 'LONDON_NY', 'spread_tier': 1},
    'AUD/JPY': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'carry_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'NZD/JPY': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'carry_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'CAD/JPY': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'carry_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'GBP/JPY': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'carry_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'EUR/JPY': {'long_char': 'MILD_RISK_ON', 'short_char': 'RISK_OFF', 'category': 'carry_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'CHF/JPY': {'long_char': 'NEUTRAL', 'short_char': 'NEUTRAL', 'category': 'carry_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'EUR/GBP': {'long_char': 'NEUTRAL', 'short_char': 'NEUTRAL', 'category': 'eur_cross', 'session': 'LONDON', 'spread_tier': 1},
    'EUR/AUD': {'long_char': 'RISK_OFF', 'short_char': 'RISK_ON', 'category': 'eur_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'EUR/NZD': {'long_char': 'RISK_OFF', 'short_char': 'RISK_ON', 'category': 'eur_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'EUR/CAD': {'long_char': 'MILD_RISK_OFF', 'short_char': 'MILD_RISK_ON', 'category': 'eur_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'EUR/CHF': {'long_char': 'MILD_RISK_ON', 'short_char': 'RISK_OFF', 'category': 'eur_cross', 'session': 'LONDON', 'spread_tier': 2},
    'GBP/AUD': {'long_char': 'NEUTRAL', 'short_char': 'RISK_ON', 'category': 'gbp_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'GBP/NZD': {'long_char': 'NEUTRAL', 'short_char': 'RISK_ON', 'category': 'gbp_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'GBP/CAD': {'long_char': 'NEUTRAL', 'short_char': 'NEUTRAL', 'category': 'gbp_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'GBP/CHF': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'gbp_cross', 'session': 'LONDON', 'spread_tier': 2},
    'AUD/NZD': {'long_char': 'NEUTRAL', 'short_char': 'NEUTRAL', 'category': 'comm_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'AUD/CAD': {'long_char': 'NEUTRAL', 'short_char': 'NEUTRAL', 'category': 'comm_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'AUD/CHF': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'comm_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'NZD/CAD': {'long_char': 'NEUTRAL', 'short_char': 'NEUTRAL', 'category': 'comm_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'NZD/CHF': {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'comm_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'CAD/CHF': {'long_char': 'MILD_RISK_ON', 'short_char': 'RISK_OFF', 'category': 'comm_cross', 'session': 'LONDON_NY', 'spread_tier': 2},
    'XAU/USD': {'long_char': 'ANTI_FIAT', 'short_char': 'RISK_ON', 'category': 'commodity', 'session': 'LONDON_NY', 'spread_tier': 1, 'always_include': True},
    'XAG/USD': {'long_char': 'MIXED', 'short_char': 'MIXED', 'category': 'commodity', 'session': 'LONDON_NY', 'spread_tier': 3},
    'WTI':     {'long_char': 'CONDITIONAL', 'short_char': 'CONDITIONAL', 'category': 'commodity', 'session': 'NYMEX', 'spread_tier': 2},
    'NAS100':  {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'index', 'session': 'US_CASH', 'spread_tier': 1},
    'SPX500':  {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'index', 'session': 'US_CASH', 'spread_tier': 1},
    'US30':    {'long_char': 'RISK_ON', 'short_char': 'RISK_OFF', 'category': 'index', 'session': 'US_CASH', 'spread_tier': 2},
}

os.makedirs(SESSION_DATA_DIR, exist_ok=True)
os.makedirs(COT_DATA_DIR, exist_ok=True)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ENGINE 1: COT PROCESSOR (unchanged from v2.2)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class COTProcessor:
    INSTRUMENTS = {
        '099741': {'name': 'EUR (6E)', 'symbol': 'EUR', 'class': 'forex', 'lookback': 26, 'report': 'tff'},
        '096742': {'name': 'GBP (6B)', 'symbol': 'GBP', 'class': 'forex', 'lookback': 26, 'report': 'tff'},
        '097741': {'name': 'JPY (6J)', 'symbol': 'JPY', 'class': 'forex', 'lookback': 26, 'report': 'tff'},
        '232741': {'name': 'AUD (6A)', 'symbol': 'AUD', 'class': 'forex', 'lookback': 26, 'report': 'tff'},
        '112741': {'name': 'NZD (6N)', 'symbol': 'NZD', 'class': 'forex', 'lookback': 26, 'report': 'tff'},
        '090741': {'name': 'CAD (6C)', 'symbol': 'CAD', 'class': 'forex', 'lookback': 26, 'report': 'tff'},
        '092741': {'name': 'CHF (6S)', 'symbol': 'CHF', 'class': 'forex', 'lookback': 26, 'report': 'tff'},
        '088691': {'name': 'Gold (GC)', 'symbol': 'XAU', 'class': 'commodity', 'lookback': 26, 'report': 'disagg'},
        '084691': {'name': 'Silver (SI)', 'symbol': 'XAG', 'class': 'commodity', 'lookback': 26, 'report': 'disagg'},
        '067651': {'name': 'WTI Crude (CL)', 'symbol': 'WTI', 'class': 'commodity', 'lookback': 13, 'report': 'disagg'},
        '209742': {'name': 'NAS E-mini (NQ)', 'symbol': 'NAS100', 'class': 'index', 'lookback': 13, 'report': 'tff'},
        '13874A': {'name': 'S&P E-mini (ES)', 'symbol': 'SPX500', 'class': 'index', 'lookback': 13, 'report': 'tff'},
        '124603': {'name': 'US30 E-mini (YM)', 'symbol': 'US30', 'class': 'index', 'lookback': 13, 'report': 'tff'},
    }
    REPORT_URLS = {
        'disagg': 'https://www.cftc.gov/files/dea/history/fut_disagg_txt_{year}.zip',
        'tff':    'https://www.cftc.gov/files/dea/history/fut_fin_txt_{year}.zip',
        'legacy': 'https://www.cftc.gov/files/dea/history/deacot{year}.zip',
    }

    def __init__(self, data_dir=None):
        self.data_dir = data_dir or COT_DATA_DIR
        os.makedirs(self.data_dir, exist_ok=True)
        self.raw = {'disagg': None, 'tff': None, 'legacy': None}
        self._cols = {'disagg': {}, 'tff': {}, 'legacy': {}}
        self._code_map = {'disagg': {}, 'tff': {}, 'legacy': {}}

    def download(self, years=None):
        if years is None:
            y = datetime.now().year
            years = [y - 1, y]
        for rtype in ('disagg', 'tff', 'legacy'):
            frames = []
            for yr in years:
                url = self.REPORT_URLS[rtype].format(year=yr)
                label = {'disagg': 'Disaggregated', 'tff': 'TFF (Financial)', 'legacy': 'Legacy Combined'}[rtype]
                print(f"  Downloading {label} {yr}...", end=" ")
                try:
                    r = requests.get(url, timeout=90)
                    r.raise_for_status()
                    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                        for name in z.namelist():
                            if name.endswith(('.txt', '.csv')):
                                with z.open(name) as f:
                                    df = pd.read_csv(f, low_memory=False)
                                    frames.append(df)
                                    df.to_csv(os.path.join(self.data_dir, f"{rtype}_{yr}.csv"), index=False)
                                    print(f"OK — {len(df)} rows")
                                break
                except Exception as e:
                    print(f"FAIL — {e}")
            if frames:
                self.raw[rtype] = pd.concat(frames, ignore_index=True)
            else:
                self._try_local_single(rtype)
        self._init_all_reports()

    def load_cached(self):
        for rtype in ('disagg', 'tff', 'legacy'):
            self._try_local_single(rtype)
        self._init_all_reports()

    def _try_local_single(self, rtype):
        prefix = f"{rtype}_"
        csvs = sorted([f for f in os.listdir(self.data_dir) if f.startswith(prefix) and f.endswith('.csv')])
        if csvs:
            frames = [pd.read_csv(os.path.join(self.data_dir, f), low_memory=False) for f in csvs]
            self.raw[rtype] = pd.concat(frames, ignore_index=True)
            print(f"  Cached {rtype}: {len(csvs)} files, {len(self.raw[rtype])} rows")
        else:
            print(f"  Cached {rtype}: no local files found")

    def _init_all_reports(self):
        for rtype in ('disagg', 'tff', 'legacy'):
            if self.raw[rtype] is not None:
                self._identify_columns(rtype)
                self._discover_codes(rtype)
        total = sum(len(df) for df in self.raw.values() if df is not None)
        print(f"  Total rows across all reports: {total}")

    def _identify_columns(self, rtype):
        df = self.raw[rtype]
        if df is None:
            return
        df.columns = [c.strip() for c in df.columns]

        def find(patterns, exclude=None):
            exclude = exclude or []
            for c in df.columns:
                cl = c.lower()
                if all(p in cl for p in patterns) and not any(x in cl for x in exclude):
                    return c
            return None

        def find_exact_prefix(prefix):
            for c in df.columns:
                if c.lower().startswith(prefix.lower()):
                    return c
            return None

        cols = {}
        date_candidates = [
            find_exact_prefix('Report_Date_as_'), find_exact_prefix('As_of_Date_In_Form_'),
            find(['report_date_as']), find(['as_of_date_in_form']), find(['as_of_date']),
            find(['report_date'], ['as_']), find(['as of', 'date']), find(['report', 'date']),
        ]
        cols['date'] = None
        for cand in date_candidates:
            if cand is not None:
                cols['date'] = cand
                break
        if cols['date'] is None:
            cols['date'] = find(['date'])

        cols['code'] = (find(['cftc_contract_market_code']) or find_exact_prefix('CFTC_Contract_Market_Code')
                        or find(['cftc', 'contract', 'market', 'code']) or find(['cftc', 'code'], ['market']))
        cols['market_name'] = (find(['market_and_exchange_names']) or find_exact_prefix('Market_and_Exchange_Names')
                               or find(['market', 'exchange', 'name']))
        cols['oi'] = (find(['open_interest_all'], ['change', 'old', '%']) or find_exact_prefix('Open_Interest_All')
                      or find(['open interest', 'all'], ['change', 'old', 'other', '%', 'percent'])
                      or find(['open interest'], ['change', 'old', '%', 'percent', 'noncommercial', 'commercial', 'nonreportable', 'managed', 'leveraged', 'dealer', 'asset', 'other']))

        if rtype == 'disagg':
            cols['spec_long'] = (find_exact_prefix('M_Money_Positions_Long_A') or find(['m_money_positions_long_all'], ['spread', 'change', 'old', '%'])
                                 or find(['m_money', 'long', 'all'], ['spread', 'change', 'old', '%']) or find(['m_money', 'long'], ['spread', 'change', 'old', '%'])
                                 or find(['managed', 'money', 'long'], ['spread', 'change', 'old', '%']))
            cols['spec_short'] = (find_exact_prefix('M_Money_Positions_Short_A') or find(['m_money_positions_short_all'], ['spread', 'change', 'old', '%'])
                                  or find(['m_money', 'short', 'all'], ['spread', 'change', 'old', '%']) or find(['m_money', 'short'], ['spread', 'change', 'old', '%'])
                                  or find(['managed', 'money', 'short'], ['spread', 'change', 'old', '%']))
        elif rtype == 'tff':
            cols['spec_long'] = (find_exact_prefix('Lev_Money_Positions_Long_A') or find(['lev_money_positions_long_all'], ['spread', 'change', 'old', '%'])
                                 or find(['lev_money', 'long', 'all'], ['spread', 'change', 'old', '%']) or find(['lev_money', 'long'], ['spread', 'change', 'old', '%'])
                                 or find(['leveraged', 'long'], ['spread', 'change', 'old', '%']))
            cols['spec_short'] = (find_exact_prefix('Lev_Money_Positions_Short_A') or find(['lev_money_positions_short_all'], ['spread', 'change', 'old', '%'])
                                  or find(['lev_money', 'short', 'all'], ['spread', 'change', 'old', '%']) or find(['lev_money', 'short'], ['spread', 'change', 'old', '%'])
                                  or find(['leveraged', 'short'], ['spread', 'change', 'old', '%']))
        else:
            cols['spec_long'] = (find(['noncommercial', 'long', 'all'], ['spread', 'change', 'old', '%', 'percent'])
                                 or find(['noncommercial', 'long'], ['spread', 'change', 'old', '%', 'percent']))
            cols['spec_short'] = (find(['noncommercial', 'short', 'all'], ['spread', 'change', 'old', '%', 'percent'])
                                  or find(['noncommercial', 'short'], ['spread', 'change', 'old', '%', 'percent']))

        self._cols[rtype] = cols
        needed = ['date', 'code', 'oi', 'spec_long', 'spec_short']
        missing = [k for k in needed if cols.get(k) is None]
        trader_label = {'disagg': 'Managed Money', 'tff': 'Leveraged Funds', 'legacy': 'Noncommercial'}[rtype]
        if missing:
            print(f"  ⚠ {rtype.upper()} — missing columns {missing} (looking for {trader_label})")
            print(f"    Available (first 40): {list(df.columns[:40])}")
        else:
            print(f"  ✓ {rtype.upper()} columns identified — trader category: {trader_label}")
            print(f"    Date col : {cols['date']}")
            print(f"    Long col : {cols['spec_long']}")
            print(f"    Short col: {cols['spec_short']}")
            if cols['date'] in df.columns:
                print(f"    Sample dates: {df[cols['date']].dropna().head(3).tolist()}")

    def _discover_codes(self, rtype):
        df = self.raw[rtype]
        cols = self._cols[rtype]
        if df is None or cols.get('code') is None:
            return
        code_col = cols['code']
        all_codes = df[code_col].astype(str).str.strip().unique()
        name_col = cols.get('market_name')
        print(f"\n  CODE DISCOVERY [{rtype.upper()}] (column: '{code_col}', {len(all_codes)} unique codes):")
        target_codes = {c: info for c, info in self.INSTRUMENTS.items() if info.get('report') == rtype}
        if rtype == 'legacy':
            target_codes = dict(self.INSTRUMENTS)
        name_keywords = {
            '099741': ['euro fx', 'euro', 'eur'], '096742': ['british pound', 'gbp', 'pound'],
            '097741': ['japanese yen', 'jpy', 'yen'], '232741': ['australian', 'aud'],
            '112741': ['new zealand', 'nzd'], '090741': ['canadian', 'cad'],
            '092741': ['swiss franc', 'chf'], '088691': ['gold'], '084691': ['silver'],
            '067651': ['crude oil', 'wti', 'light sweet'], '209742': ['nasdaq', 'e-mini nas', 'nsdq'],
            '13874A': ['s&p 500', 'e-mini s&p', 's&p', 'sp 500'],
            '124603': ['dow jones', 'djia', 'e-mini dow', 'dow']
        }
        for target_code, info in target_codes.items():
            target_str = str(target_code).strip()
            if target_str in all_codes:
                self._code_map[rtype][target_code] = target_str
                print(f"  ✓ {info['name']:<20} {target_str} → {(df[code_col].astype(str).str.strip() == target_str).sum()} rows (exact)")
                continue
            matched = False
            for actual_code in all_codes:
                if actual_code.lstrip('0') == target_str.lstrip('0'):
                    self._code_map[rtype][target_code] = actual_code
                    print(f"  ✓ {info['name']:<20} {target_str} → '{actual_code}' ({(df[code_col].astype(str).str.strip() == actual_code).sum()} rows, stripped)")
                    matched = True
                    break
            if matched:
                continue
            if name_col and name_col in df.columns:
                for kw in name_keywords.get(target_code, []):
                    name_mask = df[name_col].astype(str).str.lower().str.contains(kw, na=False)
                    if name_mask.any():
                        best = df.loc[name_mask, code_col].astype(str).str.strip().unique()[0]
                        self._code_map[rtype][target_code] = best
                        print(f"  ✓ {info['name']:<20} {target_str} → '{best}' ({(df[code_col].astype(str).str.strip() == best).sum()} rows, via name '{kw}')")
                        matched = True
                        break
            if not matched:
                print(f"  ✗ {info['name']:<20} {target_str} → NO MATCH in {rtype}")

    def _extract_series(self, rtype, code):
        if self.raw.get(rtype) is None or code not in self._code_map.get(rtype, {}):
            return None, 0
        cols = self._cols.get(rtype, {})
        needed = ['date', 'code', 'oi', 'spec_long', 'spec_short']
        if not all(cols.get(k) is not None for k in needed):
            return None, 0
        actual_code = self._code_map[rtype][code]
        df = self.raw[rtype]
        c = cols
        mask = df[c['code']].astype(str).str.strip() == actual_code
        sub = df.loc[mask, [c['date'], c['oi'], c['spec_long'], c['spec_short']]].copy()
        sub.columns = ['date', 'oi', 'spec_long', 'spec_short']
        sub['date'] = pd.to_datetime(sub['date'], errors='coerce')
        nat_count = sub['date'].isna().sum()
        if nat_count > len(sub) * 0.5 and len(sub) > 0:
            raw_dates = df.loc[mask, c['date']].copy()
            for fmt in ['%Y-%m-%d', '%m/%d/%Y', '%m/%d/%y', '%d-%b-%Y', '%Y%m%d', '%d/%m/%Y', '%B %d, %Y']:
                try:
                    parsed = pd.to_datetime(raw_dates, format=fmt, errors='coerce')
                    if parsed.notna().sum() > nat_count:
                        sub['date'] = parsed.values
                        break
                except Exception:
                    continue
        sub = sub.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
        for col in ['oi', 'spec_long', 'spec_short']:
            sub[col] = pd.to_numeric(sub[col], errors='coerce')
        sub = sub.dropna().drop_duplicates(subset='date', keep='last').reset_index(drop=True)
        sub['net'] = sub['spec_long'] - sub['spec_short']
        return sub, len(sub)

    def _get_series(self, code):
        info = self.INSTRUMENTS.get(str(code), {})
        preferred = info.get('report', 'legacy')
        lb = info.get('lookback', 26)
        order = [preferred]
        if preferred != 'legacy':
            order.append('legacy')
        best_df, best_rtype, best_count = None, None, 0
        for rtype in order:
            sub, count = self._extract_series(rtype, code)
            if sub is not None and count >= lb:
                return sub, rtype
            if count > best_count:
                best_df, best_rtype, best_count = sub, rtype, count
        return best_df, best_rtype

    def willco(self, code, lookback=None):
        info = self.INSTRUMENTS.get(str(code), {})
        lb = lookback or info.get('lookback', 26)
        name = info.get('name', code)
        symbol = info.get('symbol', code)
        asset_class = info.get('class', 'unknown')
        preferred = info.get('report', 'legacy')
        df, rtype_used = self._get_series(code)
        if df is None or len(df) < lb:
            return {'instrument': name, 'willco': None, 'asset_class': asset_class, 'report_used': None, 'is_fallback': False,
                    'error': f'Need {lb}wk, have {0 if df is None else len(df)}'}
        report_labels = {'disagg': 'Disaggregated (Managed Money)', 'tff': 'TFF (Leveraged Funds)', 'legacy': 'Legacy (Noncommercial)'}
        report_label = report_labels.get(rtype_used, rtype_used)
        is_fallback = (rtype_used != preferred)
        if is_fallback:
            report_label += ' ⚠fallback'
        df['Q'] = df['net'] / df['oi']
        window = df.tail(lb)
        q_cur, q_min, q_max = window['Q'].iloc[-1], window['Q'].min(), window['Q'].max()
        q_range = q_max - q_min
        wc = ((q_cur - q_min) / q_range * 100.0) if q_range != 0 else 50.0
        wc_156 = None
        if len(df) >= 156:
            w156 = df.tail(156)
            q156_r = w156['Q'].max() - w156['Q'].min()
            if q156_r != 0:
                wc_156 = round((w156['Q'].iloc[-1] - w156['Q'].min()) / q156_r * 100.0, 1)
        cur_long, cur_short = df['spec_long'].iloc[-1], df['spec_short'].iloc[-1]
        total_spec = cur_long + cur_short
        pct_long = (cur_long / total_spec * 100.0) if total_spec > 0 else 50.0
        pct_short = (cur_short / total_spec * 100.0) if total_spec > 0 else 50.0
        prev_net = df['net'].iloc[-2] if len(df) >= 2 else df['net'].iloc[-1]
        w4ago_net = df['net'].iloc[-4] if len(df) >= 4 else df['net'].iloc[-1]
        oi_cur = df['oi'].iloc[-1]
        oi_52 = df.tail(52)['oi'].mean() if len(df) >= 52 else df['oi'].mean()
        oi_rat = oi_cur / oi_52 if oi_52 else 0
        oi_4w_ago = df['oi'].iloc[-4] if len(df) >= 4 else df['oi'].iloc[-1]
        oi_roc_4w = ((oi_cur - oi_4w_ago) / oi_4w_ago * 100.0) if oi_4w_ago != 0 else 0.0
        signal, setup_valid = self._signal(wc, oi_rat, pct_long, pct_short, asset_class, symbol, wc_156)
        
        adx_val = COT_WEEKLY_ADX.get(symbol)
        adx_str = f"ADX: {adx_val} (needs ≤ 25)" if adx_val else "MANUAL — Check ADX ≤ 25"
        
        return {
            'instrument': name, 'asset_class': asset_class, 'report_used': report_label, 'is_fallback': is_fallback,
            'willco': round(wc, 1), 'willco_156': wc_156, 'Q_current': round(q_cur, 4), 'Q_min': round(q_min, 4), 'Q_max': round(q_max, 4),
            'net': int(df['net'].iloc[-1]), 'nc_long': int(cur_long), 'nc_short': int(cur_short),
            'pct_long': round(pct_long, 1), 'pct_short': round(pct_short, 1),
            'net_chg_1w': int(df['net'].iloc[-1] - prev_net),
            'trend_4w': 'ADDING' if df['net'].iloc[-1] > w4ago_net else 'REDUCING' if df['net'].iloc[-1] < w4ago_net else 'FLAT',
            'oi': int(oi_cur), 'oi_52w_avg': int(oi_52), 'oi_ratio': round(oi_rat, 2), 'oi_gate': oi_rat >= 1.0,
            'oi_roc_4w': round(oi_roc_4w, 2), 'oi_roc_pass': oi_roc_4w <= 0,
            'window': lb, 'net_min': int(window['net'].min()), 'net_max': int(window['net'].max()),
            'as_of': df['date'].iloc[-1].strftime('%Y-%m-%d'), 'signal': signal, 'setup_valid': setup_valid,
            'adx_check': adx_str,
        }

    @staticmethod
    def _signal(wc, oi_rat, pct_long, pct_short, asset_class, symbol, wc_156=None):
        oi_pass = oi_rat >= 1.0
        oi_tag = 'PASS' if oi_pass else 'LOW_OI'
        
        is_qe = GLOBAL_LIQUIDITY in ('EXPANSION', 'LEAN_EXPANSION')
        thresh_high = 95 if is_qe else 80
        thresh_low = 5 if is_qe else 20
        
        if asset_class == 'index':
            thresh_high = max(thresh_high, 90)
            
        adx_val = COT_WEEKLY_ADX.get(symbol)
        adx_pass = adx_val is not None and adx_val <= 25
        adx_tag = f"ADX:{adx_val}" if adx_val else "NO_ADX"
        
        tradeable = symbol not in ('AUD', 'NZD')
        
        setup_valid = False
        
        if wc >= thresh_high:
            pct_confirm = pct_long >= 75
            ctag = '%-L CONFIRMED' if pct_confirm else '%-L not confirmed'
            secular = ' | 156w SECULAR' if wc_156 is not None and wc_156 >= 80 else ''
            if pct_confirm and oi_pass and adx_pass and tradeable:
                setup_valid = True
            return f"EXTREME LONG ({wc:.0f}) › Contrarian BEAR [{oi_tag}] [{ctag}] [{adx_tag}]{secular}", setup_valid
        elif wc >= 75:
            return f"ELEVATED LONG ({wc:.0f}) › watch [{oi_tag}]", False
        elif wc <= thresh_low:
            pct_confirm = pct_short >= 75
            ctag = '%-S CONFIRMED' if pct_confirm else '%-S not confirmed'
            secular = ' | 156w SECULAR' if wc_156 is not None and wc_156 <= 20 else ''
            if pct_confirm and oi_pass and adx_pass and tradeable:
                setup_valid = True
            return f"EXTREME SHORT ({wc:.0f}) › Contrarian BULL [{oi_tag}] [{ctag}] [{adx_tag}]{secular}", setup_valid
        elif wc <= 25:
            return f"DEPRESSED ({wc:.0f}) › watch [{oi_tag}]", False
        return f"NEUTRAL ({wc:.0f}) [{oi_tag}]", False

    def all_results(self):
        return {code: self.willco(code) for code in self.INSTRUMENTS}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ENGINE 2: FRED DASHBOARD (unchanged from v2.2)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class FREDDashboard:
    SERIES = {'VIXCLS': 'VIX', 'BAMLH0A0HYM2': 'HY OAS', 'DGS2': 'US 2Y', 'DGS10': 'US 10Y',
              'DFII10': 'TIPS 10Y Real', 'T10YIE': 'Breakeven 10Y', 'DTWEXBGS': 'USD Broad TWI'}

    def __init__(self, api_key):
        self.key = api_key
        self.url = "https://api.stlouisfed.org/fred/series/observations"
        self.data = {}

    def fetch(self, sid, days=90):
        start = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
        try:
            resp = requests.get(self.url, params=dict(series_id=sid, api_key=self.key, file_type='json', observation_start=start, sort_order='desc', limit=10), timeout=15)
            resp.raise_for_status()
            obs = [o for o in resp.json().get('observations', []) if o['value'] != '.']
            if obs:
                cur = float(obs[0]['value'])
                prev = float(obs[1]['value']) if len(obs) > 1 else None
                self.data[sid] = {'val': cur, 'date': obs[0]['date'], 'prev': prev, 'chg': cur - prev if prev else None}
        except Exception as e:
            print(f"  FAIL {sid}: {e}")

    def fetch_all(self):
        print("  Fetching FRED data...")
        for sid in self.SERIES:
            self.fetch(sid)
        print(f"  Done — {len(self.data)}/{len(self.SERIES)} series loaded")

    def risk_regime(self, session_state=None):
        v = self.data.get('VIXCLS', {}).get('val')
        if v is None:
            return {'regime': 'UNKNOWN', 'vix': None, 'carry': 'UNKNOWN', 'p18_watch': False, 'p18_suspended': False, 'hy_direction': None}
        regime = 'EXTREME RISK-OFF' if v > 50 else 'SEVERE RISK-OFF' if v > 35 else 'ELEVATED RISK-OFF' if v > 25 else 'MODERATE RISK-OFF' if v > 20 else 'RISK-ON'
        carry = 'DISABLED' if v > 22 else 'ENABLED'
        hy = self.data.get('BAMLH0A0HYM2', {})
        hy_dir = ('WIDENING' if hy.get('chg', 0) > 0 else 'TIGHTENING' if hy.get('chg', 0) < 0 else 'FLAT') if hy.get('chg') is not None else None
        # P18 VIX>25 duration tracking (Finding 2)
        p18_suspended = False
        if session_state is not None:
            st = session_state.state if hasattr(session_state, 'state') else session_state
            if v > 25:
                start = st.get('vix_above_25_start_date')
                if start is None:
                    st['vix_above_25_start_date'] = datetime.now().strftime('%Y-%m-%d')
                else:
                    try:
                        days_above = (datetime.now() - datetime.strptime(start, '%Y-%m-%d')).days
                        if days_above >= 14:
                            p18_suspended = True
                    except Exception:
                        pass
            else:
                st['vix_above_25_start_date'] = None
            st['p18_suspended'] = p18_suspended
        return {'regime': regime, 'vix': v, 'carry': carry, 'p18_watch': v > 25, 'p18_suspended': p18_suspended, 'hy_direction': hy_dir}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ENGINE 3: RATE DIFFERENTIALS — ALL 28 PAIRS (CHANGE 1)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class RateDiffEngine:
    MAJORS = {
        'EUR': 'EUR/USD', 'GBP': 'GBP/USD', 'JPY': 'USD/JPY',
        'AUD': 'AUD/USD', 'NZD': 'NZD/USD', 'CAD': 'USD/CAD', 'CHF': 'USD/CHF',
    }
    CROSSES = [
        'EUR/JPY', 'EUR/GBP', 'EUR/AUD', 'EUR/CAD', 'EUR/NZD', 'EUR/CHF',
        'GBP/JPY', 'GBP/AUD', 'GBP/CAD', 'GBP/NZD', 'GBP/CHF',
        'AUD/JPY', 'NZD/JPY', 'CAD/JPY', 'CHF/JPY',
        'AUD/CAD', 'AUD/CHF', 'AUD/NZD', 'NZD/CAD', 'NZD/CHF', 'CAD/CHF',
    ]

    def __init__(self, us_2y, foreign_yields, session_dir=None):
        self.us = us_2y
        self.fy = {}
        for ccy, info in foreign_yields.items():
            self.fy[ccy] = info['yield'] if isinstance(info, dict) else info
        self.fy_meta = foreign_yields
        self.session_dir = session_dir or SESSION_DATA_DIR
        self._prior_diffs = self._load_prior()

    def _save_path(self):
        return os.path.join(self.session_dir, f"rate_diffs_{datetime.now().strftime('%Y-%m-%d')}.json")

    def _load_prior(self):
        try:
            files = sorted([f for f in os.listdir(self.session_dir) if f.startswith('rate_diffs_') and f.endswith('.json')])
            today_file = f"rate_diffs_{datetime.now().strftime('%Y-%m-%d')}.json"
            prior_files = [f for f in files if f != today_file]
            if prior_files:
                with open(os.path.join(self.session_dir, prior_files[-1]), 'r') as fh:
                    print(f"  Loaded prior rate diffs from {prior_files[-1]}")
                    return json.load(fh)
        except Exception:
            pass
        return {}

    def _save_current(self, diffs):
        try:
            save_data = {pair: {'spread_bp': d.get('spread_bp'), 'date': datetime.now().strftime('%Y-%m-%d')} for pair, d in diffs.items()}
            with open(self._save_path(), 'w') as fh:
                json.dump(save_data, fh, indent=2)
        except Exception as e:
            print(f"  ⚠ Could not save rate diffs: {e}")

    def _is_stale(self, ccy):
        info = self.fy_meta.get(ccy)
        if isinstance(info, dict) and info.get('updated', 'unknown') != 'unknown':
            try:
                return (datetime.now() - datetime.strptime(info['updated'], '%Y-%m-%d')).days > 7
            except Exception:
                return True
        return False

    def _calc_spread(self, pair, base_yield, quote_yield):
        sp = base_yield - quote_yield
        prior = self._prior_diffs.get(pair, {})
        prior_spread = prior.get('spread_bp')
        shift_bp, shift_class = None, None
        if prior_spread is not None:
            shift_bp = round(sp * 100 - prior_spread, 1)
            abs_shift = abs(shift_bp)
            shift_class = 'STRONG' if abs_shift >= 50 else 'MODERATE' if abs_shift >= 25 else 'LOW'
        strength = 'MAX' if abs(sp) > 2.0 else 'STRONG' if abs(sp) > 1.0 else 'MOD' if abs(sp) > 0.5 else 'LOW'
        parts = pair.split('/')
        base_ccy, quote_ccy = parts[0], parts[1]
        favors = base_ccy if sp > 0 else quote_ccy
        return {
            'spread_bp': round(sp * 100, 1), 'base_yield': base_yield, 'quote_yield': quote_yield,
            'favors': favors, 'strength': strength, 'shift_bp': shift_bp, 'shift_class': shift_class,
            'last_assessed': prior.get('date', '—'), 'pair_type': 'major' if pair in self.MAJORS.values() else 'cross',
        }

    def spreads(self):
        out = {}
        for ccy, pair in self.MAJORS.items():
            fy_val = self.fy.get(ccy)
            if fy_val is None:
                out[pair] = {'spread_bp': None, 'note': 'MISSING DATA', 'pair_type': 'major'}
                continue
            parts = pair.split('/')
            if parts[0] == 'USD':
                base_y, quote_y = self.us, fy_val
            else:
                base_y, quote_y = fy_val, self.us
            result = self._calc_spread(pair, base_y, quote_y)
            result['stale'] = self._is_stale(ccy)
            out[pair] = result
        for pair in self.CROSSES:
            parts = pair.split('/')
            base_ccy, quote_ccy = parts[0], parts[1]
            base_y = self.fy.get(base_ccy)
            quote_y = self.fy.get(quote_ccy)
            if base_y is None or quote_y is None:
                out[pair] = {'spread_bp': None, 'note': 'MISSING DATA', 'pair_type': 'cross'}
                continue
            result = self._calc_spread(pair, base_y, quote_y)
            result['stale'] = self._is_stale(base_ccy) or self._is_stale(quote_ccy)
            out[pair] = result
        self._save_current(out)
        return out

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ENGINE 4: EVENT CALENDAR — Forex Factory API (FIX 3+4: country-awareness + ADP/NFP)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class EventCalendar:
    FF_URL = "https://nfs.faireconomy.media/ff_calendar_thisweek.json"

    # ── FIX 3A: Country-filtered EVENT_MAP ──
    # Added optional 'countries' field to US-specific entries.
    # When present, the entry only matches if the FF API item's country is in the list.
    # FIX 3E: US CPI instruments corrected per Master Event System Section 3.
    # FIX 4: ADP entry placed BEFORE NFP to prevent keyword collision.
    EVENT_MAP = [
        (['FOMC Statement', 'Federal Funds Rate', 'FOMC Economic Projections'],
         {'event': 'FOMC Decision', 'tier': 1, 'pre': 120, 'post': 240, 'instruments': ['ALL']}),
        # FIX 4: ADP Employment — MUST be before NFP entry.
        # FF API title "ADP Non-Farm Employment Change" contains "Non-Farm Employment Change",
        # so without this entry it would match NFP's keywords and get Tier 1 / 30 / 180 / ALL.
        # Placed here, ADP matches first on its more-specific keywords.
        # BLS title "Non-Farm Employment Change" (without "ADP" prefix) will NOT match these
        # keywords and correctly falls through to the NFP entry below.
        (['ADP Non-Farm Employment Change', 'ADP Nonfarm Employment Change'],
         {'event': 'ADP Employment', 'tier': 2, 'pre': 15, 'post': 15,
          'instruments': ['EUR/USD', 'GBP/USD', 'USD/JPY', 'XAU/USD']}),
        (['Non-Farm Employment Change'],
         {'event': 'NFP', 'tier': 1, 'pre': 30, 'post': 180, 'instruments': ['ALL']}),
        # FIX 3A+3E: CPI — country-filtered to USD only; instruments scoped per Section 3
        (['Core CPI m/m', 'CPI m/m', 'CPI y/y'],
         {'event': 'CPI', 'tier': 1, 'pre': 30, 'post': 120,
          'instruments': ['EUR/USD', 'GBP/USD', 'USD/JPY', 'XAU/USD', 'NAS100', 'SPX500', 'US30'],
          'countries': ['USD']}),
        # FIX 3A+3E: PCE — country-filtered to USD; instruments scoped per Section 3
        (['Core PCE Price Index m/m', 'Core PCE Price Index y/y', 'PCE Price Index m/m', 'PCE Price Index y/y'],
         {'event': 'PCE', 'tier': 1, 'pre': 30, 'post': 120,
          'instruments': ['EUR/USD', 'GBP/USD', 'USD/JPY', 'XAU/USD', 'NAS100', 'SPX500', 'US30'],
          'countries': ['USD']}),
        (['Main Refinancing Rate', 'ECB Press Conference'],
         {'event': 'ECB Decision', 'tier': 1, 'pre': 30, 'post': 120,
          'instruments': ['EUR/USD', 'EUR/GBP', 'EUR/JPY', 'EUR/CHF', 'XAU/USD']}),
        (['BOE Monetary Policy Report'],
         {'event': 'BoE Decision + MPR', 'tier': 1, 'pre': 30, 'post': 120,
          'instruments': ['GBP/USD', 'EUR/GBP', 'GBP/JPY']}),
        (['Official Bank Rate'],
         {'event': 'BoE Decision', 'tier': 1, 'pre': 30, 'post': 60,
          'instruments': ['GBP/USD', 'EUR/GBP', 'GBP/JPY']}),
        (['BOJ Policy Rate', 'BOJ Press Conference'],
         {'event': 'BoJ Decision', 'tier': 1, 'pre': 60, 'post': 120,
          'instruments': ['USD/JPY', 'AUD/JPY', 'NZD/JPY', 'EUR/JPY']}),
        (['Overnight Rate', 'BOC Rate Statement'],
         {'event': 'BoC Decision', 'tier': 1, 'pre': 30, 'post': 60,
          'instruments': ['USD/CAD', 'EUR/CAD', 'CAD/JPY']}),
        (['Cash Rate', 'RBA Rate Statement'],
         {'event': 'RBA Decision', 'tier': 1, 'pre': 30, 'post': 60,
          'instruments': ['AUD/USD', 'AUD/NZD', 'AUD/JPY']}),
        (['Official Cash Rate', 'RBNZ Rate Statement'],
         {'event': 'RBNZ Decision', 'tier': 1, 'pre': 30, 'post': 60,
          'instruments': ['NZD/USD', 'AUD/NZD', 'NZD/JPY', 'GBP/NZD']}),
        (['Crude Oil Inventories'],
         {'event': 'EIA Crude Inventory', 'tier': 2, 'pre': 60, 'post': 30, 'instruments': ['WTI']}),
        (['OPEC-JMMC Meetings', 'OPEC Meetings'],
         {'event': 'OPEC+ Meeting', 'tier': 1, 'pre': 1440, 'post': 1440, 'instruments': ['WTI']}),
        (['ISM Services PMI'],
         {'event': 'ISM Services PMI', 'tier': 2, 'pre': 30, 'post': 30, 'instruments': ['ALL']}),
        (['ISM Manufacturing PMI'],
         {'event': 'ISM Manufacturing PMI', 'tier': 2, 'pre': 30, 'post': 30, 'instruments': ['ALL']}),
        # FIX 3A: Retail Sales — country-filtered to USD
        (['Retail Sales m/m', 'Core Retail Sales m/m'],
         {'event': 'Retail Sales', 'tier': 2, 'pre': 30, 'post': 30, 'instruments': ['ALL'],
          'countries': ['USD']}),
        # FIX 3A: GDP — country-filtered to USD
        (['Advance GDP q/q'],
         {'event': 'GDP Advance', 'tier': 2, 'pre': 30, 'post': 30, 'instruments': ['ALL'],
          'countries': ['USD']}),
        # FIX 3A: PPI — country-filtered to USD
        (['PPI m/m', 'Core PPI m/m'],
         {'event': 'PPI', 'tier': 2, 'pre': 30, 'post': 30, 'instruments': ['ALL'],
          'countries': ['USD']}),
        # FIX 3A: Jobless Claims — country-filtered to USD
        (['Unemployment Claims'],
         {'event': 'Jobless Claims', 'tier': 2, 'pre': 15, 'post': 15, 'instruments': ['ALL'],
          'countries': ['USD']}),
    ]

    # ── FIX 3B: Non-US data release mapping ──
    # Used when a country-filtered US entry's keywords match but the country is NOT USD.
    # These are Tier 2 with smaller blackout windows, scoped to that country's pairs.
    NON_US_DATA_MAP = {
        'CPI':          {'tier': 2, 'pre': 15, 'post': 30},
        'PCE':          {'tier': 2, 'pre': 15, 'post': 30},
        'PPI':          {'tier': 2, 'pre': 15, 'post': 15},
        'Retail Sales': {'tier': 2, 'pre': 15, 'post': 15},
        'GDP Advance':  {'tier': 2, 'pre': 15, 'post': 15},
        'Jobless Claims': {'tier': 2, 'pre': 15, 'post': 15},
    }

    # ── FIX 3C: Country → affected instruments mapping ──
    # When a non-US data release matches, use this instead of ['ALL'].
    COUNTRY_INSTRUMENTS = {
        'USD': ['ALL'],  # handled by primary EVENT_MAP
        'EUR': ['EUR/USD', 'EUR/GBP', 'EUR/JPY', 'EUR/CHF'],
        'GBP': ['GBP/USD', 'EUR/GBP', 'GBP/JPY'],
        'JPY': ['USD/JPY', 'EUR/JPY', 'AUD/JPY', 'NZD/JPY'],
        'AUD': ['AUD/USD', 'AUD/NZD', 'AUD/JPY'],
        'NZD': ['NZD/USD', 'AUD/NZD', 'NZD/JPY', 'GBP/NZD'],
        'CAD': ['USD/CAD', 'EUR/CAD', 'CAD/JPY'],
        'CHF': ['USD/CHF', 'EUR/CHF'],
    }

    DEFAULT_BLACKOUTS = {
        'FOMC': {'pre': 120, 'post': 240, 'instruments': ['ALL']},
        'NFP': {'pre': 30, 'post': 180, 'instruments': ['ALL']},
        'CPI': {'pre': 30, 'post': 120, 'instruments': ['EUR/USD', 'GBP/USD', 'USD/JPY', 'XAU/USD', 'NAS100', 'SPX500', 'US30']},
        'PCE': {'pre': 30, 'post': 120, 'instruments': ['EUR/USD', 'GBP/USD', 'USD/JPY', 'XAU/USD', 'NAS100', 'SPX500', 'US30']},
        'ECB': {'pre': 30, 'post': 120, 'instruments': ['EUR/USD', 'EUR/GBP', 'EUR/JPY', 'EUR/CHF', 'XAU/USD']},
        'BoE + MPR': {'pre': 30, 'post': 120, 'instruments': ['GBP/USD', 'EUR/GBP', 'GBP/JPY']},
        'BoE': {'pre': 30, 'post': 60, 'instruments': ['GBP/USD', 'EUR/GBP', 'GBP/JPY']},
        'BoJ': {'pre': 60, 'post': 120, 'instruments': ['USD/JPY', 'AUD/JPY', 'NZD/JPY', 'EUR/JPY']},
        'BoC': {'pre': 30, 'post': 60, 'instruments': ['USD/CAD', 'EUR/CAD', 'CAD/JPY']},
        'RBA': {'pre': 30, 'post': 60, 'instruments': ['AUD/USD', 'AUD/NZD', 'AUD/JPY']},
        'RBNZ': {'pre': 30, 'post': 60, 'instruments': ['NZD/USD', 'AUD/NZD', 'NZD/JPY', 'GBP/NZD']},
        'EIA': {'pre': 60, 'post': 30, 'instruments': ['WTI']},
        'OPEC': {'pre': 1440, 'post': 1440, 'instruments': ['WTI']},
        'ISM': {'pre': 30, 'post': 30, 'instruments': ['ALL']},
        'Retail Sales': {'pre': 30, 'post': 30, 'instruments': ['ALL']},
        'GDP': {'pre': 30, 'post': 30, 'instruments': ['ALL']},
        'PPI': {'pre': 30, 'post': 30, 'instruments': ['ALL']},
        'API': {'pre': 30, 'post': 30, 'instruments': ['WTI']},
        'Jobless': {'pre': 15, 'post': 15, 'instruments': ['ALL']},
    }

    def __init__(self, manual_events=None):
        self.manual_events = manual_events or []
        self._processed = []
        self._warnings = []
        self._fetch_and_process()

    def _fetch_ff(self):
        """FIX 3D: Country-aware matching logic.
        FIX 4: ADP/NFP collision resolved by EVENT_MAP ordering.

        For each FF API item:
        1. Try matching against EVENT_MAP entries by keyword (iterated in order, first match wins).
        2. If matched entry has 'countries' field, check if item's country is in the list.
           - If country check PASSES → use US-specific tier/pre/post/instruments (primary match).
           - If country check FAILS → fall through to NON_US_DATA_MAP using the matched base_event name.
        3. If NON_US_DATA_MAP matches → use its tier/pre/post + COUNTRY_INSTRUMENTS for affected pairs.
        4. If no match anywhere → skip the event (current behavior).
        """
        events = []
        try:
            resp = requests.get(self.FF_URL, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}, timeout=15)
            resp.raise_for_status()
            ff_data = resp.json()
            seen_ids = set()
            non_us_matched = 0  # diagnostic counter
            for item in ff_data:
                if item.get('impact', '').lower() == 'low':
                    continue
                title = item.get('title', '')
                country = item.get('country', '').strip().upper()

                # Step 1: Try matching against EVENT_MAP keywords
                matched_mapping = None
                matched_keywords = False
                for keywords, mapping in self.EVENT_MAP:
                    if any(kw.lower() in title.lower() for kw in keywords):
                        matched_keywords = True
                        # Step 2: Check country filter if present
                        allowed_countries = mapping.get('countries')
                        if allowed_countries is not None:
                            # This entry has a country filter
                            if country in allowed_countries:
                                # Country matches → use this US-specific mapping
                                matched_mapping = mapping
                                break
                            else:
                                # Country does NOT match (e.g., NZ CPI, UK PPI)
                                # → Do NOT use this US mapping; fall through to NON_US_DATA_MAP
                                # Store the base_event name for NON_US_DATA_MAP lookup
                                base_event_for_nonus = mapping['event']

                                # Step 3: Check NON_US_DATA_MAP
                                nonus = self.NON_US_DATA_MAP.get(base_event_for_nonus)
                                if nonus is not None:
                                    # Build a non-US event entry
                                    nonus_instruments = self.COUNTRY_INSTRUMENTS.get(country, ['ALL'])
                                    matched_mapping = {
                                        'event': base_event_for_nonus,
                                        'tier': nonus['tier'],
                                        'pre': nonus['pre'],
                                        'post': nonus['post'],
                                        'instruments': nonus_instruments,
                                        '_is_nonus': True,  # internal flag for display
                                    }
                                    non_us_matched += 1
                                    break
                                else:
                                    # No NON_US_DATA_MAP entry either → skip entirely
                                    break
                        else:
                            # No country filter on this entry → universal match (FOMC, NFP, ADP, ISM, etc.)
                            matched_mapping = mapping
                            break

                if matched_mapping is None:
                    continue

                dt_str_raw = item.get('date', '')
                try:
                    dt_str = dt_str_raw[:16]
                except Exception:
                    continue

                # Build display event name with country prefix
                base_event = matched_mapping['event']
                display_event = f"{country} {base_event}" if country else base_event

                unique_id = f"{display_event}_{dt_str}"
                if unique_id in seen_ids:
                    continue
                seen_ids.add(unique_id)

                events.append({
                    'datetime_str': dt_str.replace('T', ' '),
                    'event': display_event,
                    'base_event': base_event,
                    'country': country,
                    'tier': matched_mapping['tier'],
                    'blackout_pre_min': matched_mapping['pre'],
                    'blackout_post_min': matched_mapping['post'],
                    'instruments': matched_mapping['instruments'],
                    'source': 'FF_API',
                })
            print(f"  FF API: {len(events)} events matched from {len(ff_data)} raw entries ({non_us_matched} non-US data releases)")
        except Exception as e:
            print(f"  ⚠ FF API fetch failed: {e} — using manual events only")
        return events

    def _apply_defaults(self, ev):
        if 'blackout_pre_min' not in ev or 'blackout_post_min' not in ev:
            ev_name = ev.get('event', '')
            for key, defaults in self.DEFAULT_BLACKOUTS.items():
                if key.lower() in ev_name.lower():
                    ev.setdefault('blackout_pre_min', defaults['pre'])
                    ev.setdefault('blackout_post_min', defaults['post'])
                    ev.setdefault('instruments', defaults['instruments'])
                    ev.setdefault('tier', 1)
                    break
        ev.setdefault('blackout_pre_min', 30)
        ev.setdefault('blackout_post_min', 30)
        ev.setdefault('instruments', ['ALL'])
        ev.setdefault('tier', 2)
        ev.setdefault('base_event', ev.get('event', ''))
        ev.setdefault('country', '')
        return ev

    def _fetch_and_process(self):
        now = datetime.now()
        ff_events = self._fetch_ff()
        manual_with_defaults = [self._apply_defaults(dict(ev)) for ev in self.manual_events]
        all_events = ff_events + manual_with_defaults
        self._processed = []
        for ev in all_events:
            try:
                dt = datetime.strptime(ev['datetime_str'], '%Y-%m-%d %H:%M')
                bstart = dt - timedelta(minutes=ev['blackout_pre_min'])
                bend = dt + timedelta(minutes=ev['blackout_post_min'])
                in_bo = bstart <= now <= bend
                if now < bstart:
                    status = 'UPCOMING'
                    cd_str = f"Blackout in {self._fmt_td(bstart - now)}"
                elif in_bo:
                    status = 'BLACKOUT'
                    cd_str = f"ACTIVE — {self._fmt_td(bend - now)} remaining"
                else:
                    status = 'PASSED'
                    cd_str = "Passed"
                self._processed.append({
                    'event': ev['event'],
                    'base_event': ev.get('base_event', ev['event']),
                    'country': ev.get('country', ''),
                    'datetime': dt, 'datetime_str': ev['datetime_str'],
                    'tier': ev['tier'], 'instruments': ev['instruments'],
                    'blackout_pre_min': ev['blackout_pre_min'], 'blackout_post_min': ev['blackout_post_min'],
                    'status': status, 'in_blackout': in_bo, 'countdown_str': cd_str,
                    'source': ev.get('source', 'MANUAL'),
                })
            except Exception as e:
                self._processed.append({'event': ev.get('event', 'ERROR'), 'base_event': '', 'country': '',
                                        'status': 'ERROR', 'in_blackout': False, 'countdown_str': f'Error: {e}'})
        self._detect_compound_and_overlap()

    def _detect_compound_and_overlap(self):
        self._warnings = []
        api_events = [e for e in self._processed if 'API' in e.get('event', '').upper()]
        eia_events = [e for e in self._processed if 'EIA' in e.get('event', '').upper()]
        for api_ev in api_events:
            for eia_ev in eia_events:
                if 'datetime' in api_ev and 'datetime' in eia_ev:
                    gap = abs((eia_ev['datetime'] - api_ev['datetime']).total_seconds())
                    if gap <= 86400:
                        self._warnings.append({
                            'type': 'COMPOUND_OIL',
                            'text': f"⚠ OIL COMPOUND BLACKOUT: API + EIA within 24h — No oil trades from API pre-blackout through EIA post-blackout",
                        })
        # FIX 2 (preserved): Tier 1 overlap — only flag DIFFERENT base_event types within 48h
        tier1 = sorted([e for e in self._processed if e.get('tier') == 1 and 'datetime' in e], key=lambda x: x['datetime'])
        for i in range(len(tier1)):
            for j in range(i + 1, len(tier1)):
                if tier1[i].get('base_event', '') == tier1[j].get('base_event', ''):
                    continue
                gap = abs((tier1[j]['datetime'] - tier1[i]['datetime']).total_seconds())
                if gap <= 172800:
                    self._warnings.append({
                        'type': 'TIER1_OVERLAP',
                        'text': f"⚠ TIER 1 OVERLAP: {tier1[i]['event']} and {tier1[j]['event']} within 48h — trade higher-priority only or halve size. Priority: FOMC > NFP > CPI > ECB/BoE/BoJ > OPEC",
                    })

    @staticmethod
    def _fmt_td(td):
        total = int(td.total_seconds())
        h, m = total // 3600, (total % 3600) // 60
        return f"{h}h {m}m" if h > 0 else f"{m}m"

    def active_blackouts(self):
        blocked = set()
        for ev in self._processed:
            if ev.get('in_blackout'):
                for inst in ev.get('instruments', []):
                    blocked.add(inst)
        return blocked

    def get_events(self):
        return self._processed

    def get_warnings(self):
        return self._warnings

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ENGINE 5: SESSION STATE PERSISTENCE (unchanged)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class SessionState:
    def __init__(self, session_dir=None):
        self.session_dir = session_dir or SESSION_DATA_DIR
        os.makedirs(self.session_dir, exist_ok=True)
        self.state = self._load_latest()
        self._check_macro_staleness()

    def _save_path(self):
        return os.path.join(self.session_dir, f"session_state_{datetime.now().strftime('%Y-%m-%d')}.json")

    def _load_latest(self):
        try:
            files = sorted([f for f in os.listdir(self.session_dir) if f.startswith('session_state_') and f.endswith('.json')])
            if files:
                with open(os.path.join(self.session_dir, files[-1]), 'r') as fh:
                    state = json.load(fh)
                    print(f"  Loaded session state from {files[-1]}")
                    return state
        except Exception as e:
            print(f"  Could not load session state: {e}")
        return self._default_state()

    def _default_state(self):
        return {'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M'), 'open_positions': [], 'session_pnl_r': 0.0,
                'weekly_pnl_r': 0.0, 'daily_risk_used_pct': 0.0, 'daily_risk_cap_pct': 3.0, 'macro_assessment_date': None,
                'liquidity_regime': 'UNKNOWN', 'liquidity_score': 0, 'active_cot_signals': [], 'notes': '',
                'vix_above_25_start_date': None, 'p18_suspended': False}

    def _check_macro_staleness(self):
        macro_date = self.state.get('macro_assessment_date')
        if macro_date:
            try:
                days_old = (datetime.now() - datetime.strptime(macro_date, '%Y-%m-%d')).days
                if days_old > 30:
                    print(f"  ⚠ [WARNING: Overall Macro Assessment is > 30 days stale ({days_old} days old). Run Deep Search refresh.]")
            except Exception:
                pass

    def save(self):
        self.state['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M')
        try:
            with open(self._save_path(), 'w') as fh:
                json.dump(self.state, fh, indent=2, default=str)
            print(f"  Session state saved")
        except Exception as e:
            print(f"  ⚠ Could not save: {e}")

    def update(self, **kwargs):
        self.state.update(kwargs)

    def add_position(self, instrument, direction, entry, stop, protocol, r_status='0R'):
        self.state['open_positions'].append({'instrument': instrument, 'direction': direction, 'entry': entry, 'stop': stop,
                                             'protocol': protocol, 'r_status': r_status, 'opened': datetime.now().strftime('%Y-%m-%d %H:%M')})

    def get_positions(self):
        return self.state.get('open_positions', [])

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FISCAL DERIVATION HELPERS (CHANGE 3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INTENSITY_RANK = {None: 0, 'MILD': 1, 'MODERATE': 2, 'STRONG': 3}

def _fb_get(ccy):
    """Read FISCAL_BIAS entry. Supports both old tuple format and new dict format."""
    entry = FISCAL_BIAS.get(ccy)
    if entry is None:
        return ('NEUTRAL', None)
    if isinstance(entry, dict):
        return (entry.get('direction', 'NEUTRAL'), entry.get('intensity'))
    return entry  # legacy tuple format

def derive_pair_fiscal(pair):
    parts = pair.split('/')
    if len(parts) != 2:
        if pair in ('NAS100', 'SPX500', 'US30'):
            usd_dir, usd_int = _fb_get('USD')
            if usd_dir == 'BEARISH':
                return 'BEARISH'
            return 'NEUTRAL'
        if pair == 'WTI':
            usd_dir, _ = _fb_get('USD')
            return 'BULLISH' if usd_dir == 'BEARISH' else 'NEUTRAL'
        return 'NEUTRAL'

    base_ccy, quote_ccy = parts[0], parts[1]

    if base_ccy in ('XAU', 'XAG'):
        usd_dir, usd_int = _fb_get('USD')
        result = 'NEUTRAL'
        if usd_dir == 'BEARISH':
            result = 'BULLISH'
        elif usd_dir == 'BULLISH':
            result = 'BEARISH'
        for ccy in ('EUR', 'GBP', 'JPY'):
            fb_dir, fb_int = _fb_get(ccy)
            if fb_dir == 'BEARISH' and fb_int == 'STRONG':
                result = 'BULLISH'
                break
        return result

    base_dir, base_int = _fb_get(base_ccy)
    quote_dir, quote_int = _fb_get(quote_ccy)
    base_rank = INTENSITY_RANK.get(base_int, 0)
    quote_rank = INTENSITY_RANK.get(quote_int, 0)

    if base_dir == 'NEUTRAL' and quote_dir == 'NEUTRAL':
        return 'NEUTRAL'
    if base_dir == 'BULLISH' and quote_dir == 'NEUTRAL':
        return 'BULLISH'
    if base_dir == 'NEUTRAL' and quote_dir == 'BULLISH':
        return 'BEARISH'
    if base_dir == 'BEARISH' and quote_dir == 'NEUTRAL':
        return 'BEARISH'
    if base_dir == 'NEUTRAL' and quote_dir == 'BEARISH':
        return 'BULLISH'
    if base_dir == 'BULLISH' and quote_dir == 'BEARISH':
        return 'BULLISH'
    if base_dir == 'BEARISH' and quote_dir == 'BULLISH':
        return 'BEARISH'
    if base_dir == quote_dir:
        if base_rank == quote_rank:
            return 'NEUTRAL'
        if base_dir == 'BULLISH':
            return 'BULLISH' if base_rank > quote_rank else 'BEARISH'
        else:  # both BEARISH: higher rank = MORE bearish = WORSE for base
            return 'BEARISH' if base_rank > quote_rank else 'BULLISH'
    return 'NEUTRAL'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RETAIL SENTIMENT HELPER — Target 2
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def _rs_get(pair):
    """Safely extracts retail dictionary. Applies staleness checks and JPY gate."""
    entry = RETAIL_SENTIMENT.get(pair, {'signal': 'NO_SIGNAL', 'tier': 'STANDARD', 'sources': 0, 'updated': None})
    result = dict(entry)  # shallow copy to avoid mutating config
    
    result['stale'] = False
    updated_str = result.get('updated')
    if not updated_str:
        result['stale'] = True
    else:
        try:
            days_old = (datetime.now() - datetime.strptime(updated_str, '%Y-%m-%d')).days
            if days_old > 1:
                result['stale'] = True
        except Exception:
            result['stale'] = True
            
    # JPY Gate
    if 'JPY' in pair and result.get('sources', 0) < 2:
        result['signal'] = 'NO_SIGNAL'
        
    return result

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CLASSIFY_LAYER HELPER — FIX 1 (new function)
# Normalizes each conviction layer to SUPPORTS / OPPOSES / NEUTRAL
# relative to a specific (pair, direction) trade using INSTRUMENT_DNA.
#
# v2.3.4 FIX (Item #4): Added SEVERE_RISK_OFF to risk-off classification
# group. All three risk-off severity levels (MODERATE, SEVERE, EXTREME)
# use identical classification logic: haven trades get SUPPORTS, risk-on
# trades get OPPOSES. Severity differentiation is handled by
# REGIME_PERMISSIONS (what you're allowed to trade) and sizing (how much),
# not by conviction classification (directional alignment).
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RISK_ON_CHARS = {'RISK_ON', 'MILD_RISK_ON'}
RISK_OFF_CHARS = {'RISK_OFF', 'MILD_RISK_OFF', 'ANTI_FIAT'}

def classify_layer(layer_name, layer_value, pair, direction):
    """
    Returns 'SUPPORTS', 'OPPOSES', or 'NEUTRAL'.
    Uses the instrument's directional risk character from INSTRUMENT_DNA
    to determine whether a macro layer value supports the proposed trade.
    """
    dna = INSTRUMENT_DNA.get(pair)
    if dna is None:
        return 'NEUTRAL'
    trade_char = dna['long_char'] if direction == 'LONG' else dna['short_char']

    # ── LIQUIDITY ──
    if layer_name == 'liquidity':
        if trade_char == 'NEUTRAL':
            return 'NEUTRAL'
        if layer_value in ('EXPANSION', 'LEAN_EXPANSION'):
            if trade_char in RISK_ON_CHARS:
                return 'SUPPORTS'
            if trade_char in RISK_OFF_CHARS:
                return 'OPPOSES'
            return 'NEUTRAL'
        if layer_value in ('CONTRACTION', 'CRISIS'):
            if trade_char in RISK_OFF_CHARS:
                return 'SUPPORTS'
            if trade_char in RISK_ON_CHARS:
                return 'OPPOSES'
            return 'NEUTRAL'
        if layer_value == 'LEAN_CONTRACTION':
            if trade_char in RISK_OFF_CHARS:
                return 'SUPPORTS'
            if trade_char in RISK_ON_CHARS:
                return 'OPPOSES'
            return 'NEUTRAL'
        return 'NEUTRAL'

    # ── FISCAL ──
    if layer_name == 'fiscal':
        if layer_value == 'BULLISH':
            return 'SUPPORTS' if direction == 'LONG' else 'OPPOSES'
        if layer_value == 'BEARISH':
            return 'SUPPORTS' if direction == 'SHORT' else 'OPPOSES'
        return 'NEUTRAL'

    # ── RATE DIFF ──
    if layer_name == 'rate_diff':
        if layer_value in ('NEUTRAL', 'UNKNOWN'):
            return 'NEUTRAL'
        if layer_value.startswith('BULL_'):
            favored_ccy = layer_value[5:]
            parts = pair.split('/')
            if len(parts) == 2:
                base_ccy, quote_ccy = parts[0], parts[1]
                if (favored_ccy == base_ccy and direction == 'LONG') or \
                   (favored_ccy == quote_ccy and direction == 'SHORT'):
                    return 'SUPPORTS'
                if (favored_ccy == base_ccy and direction == 'SHORT') or \
                   (favored_ccy == quote_ccy and direction == 'LONG'):
                    return 'OPPOSES'
            return 'NEUTRAL'
        return 'NEUTRAL'

    # ── RISK SENTIMENT ──
    # v2.3.4 FIX (Item #4): SEVERE_RISK_OFF added to risk-off group.
    # Without this, SEVERE falls through to the default 'NEUTRAL' return,
    # making every instrument's risk_sentiment layer show NEUTRAL classification.
    # That artificially inflates conviction by removing OPPOSES from risk-on trades
    # and removing SUPPORTS from haven trades — exactly backward from intent.
    if layer_name == 'risk_sentiment':
        if trade_char == 'NEUTRAL':
            return 'NEUTRAL'
        if layer_value == 'RISK_ON':
            if trade_char in RISK_ON_CHARS:
                return 'SUPPORTS'
            if trade_char in RISK_OFF_CHARS:
                return 'OPPOSES'
            return 'NEUTRAL'
        if layer_value in ('MODERATE_RISK_OFF', 'SEVERE_RISK_OFF', 'EXTREME_RISK_OFF'):
            if trade_char in RISK_OFF_CHARS:
                return 'SUPPORTS'
            if trade_char in RISK_ON_CHARS:
                return 'OPPOSES'
            return 'NEUTRAL'
        return 'NEUTRAL'  # NEUTRAL sentiment

    # ── RETAIL ──
    if layer_name == 'retail':
        if layer_value == 'CONTRARIAN_BULL':
            return 'SUPPORTS' if direction == 'LONG' else 'OPPOSES'
        if layer_value == 'CONTRARIAN_BEAR':
            return 'SUPPORTS' if direction == 'SHORT' else 'OPPOSES'
        return 'NEUTRAL'

    # ── REGIME (P16) ──
    if layer_name == 'regime':
        if layer_value == 'TRENDING':
            return 'SUPPORTS'
        if layer_value == 'RANGING':
            return 'NEUTRAL'
        return 'NEUTRAL'  # NOT_ASSESSED → not populated (handled upstream)

    return 'NEUTRAL'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ENGINE 6: CONVICTION MATRIX (FIX 1 — fully rewritten logic)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class ConvictionMatrix:
    EMPTY_VALUES = {'—', 'UNKNOWN', 'NO_SIGNAL', '', 'NOT_ASSESSED', 'AUTO'}

    def __init__(self, rate_diffs=None, risk_sentiment=None, selected_instruments=None):
        """
        FIX 1: Now accepts selected_instruments as list of (pair, direction) tuples
        instead of selected_pairs (list of strings).
        """
        self.rate_diffs = rate_diffs or {}
        self.risk_sentiment = risk_sentiment or 'NEUTRAL'
        self.selected_instruments = selected_instruments or []

    def _rate_diff_direction(self, pair):
        """Hybrid Model v2.0: Directional bias uses shift_class (spread change),
        not absolute strength. Absolute strength retained for carry viability (Part 2)."""
        rd = self.rate_diffs.get(pair, {})
        if rd.get('spread_bp') is None:
            return 'UNKNOWN'
        favors = rd.get('favors', 'UNKNOWN')
        # P18 Hybrid: use shift_class for directional conviction
        shift_class = rd.get('shift_class')
        if shift_class == 'STRONG':  # shift >= 50bp
            return f"BULL_{favors}"
        # shift_class MODERATE (25-50bp) or LOW (<25bp) or None (no prior data) → NEUTRAL
        return 'NEUTRAL'

    def build(self):
        matrix = {}
        for pair, direction in self.selected_instruments:
            fiscal = derive_pair_fiscal(pair)
            retail_data = _rs_get(pair)
            retail_sig = retail_data.get('signal', 'NO_SIGNAL')
            regime_info = INSTRUMENT_REGIME.get(pair, {})
            regime_val = regime_info.get('regime', 'NOT_ASSESSED')

            # Raw layer values (for display)
            layers = {
                'liquidity': GLOBAL_LIQUIDITY,
                'fiscal': fiscal if fiscal != 'NEUTRAL' else 'NEUTRAL',
                'rate_diff': self._rate_diff_direction(pair),
                'risk_sentiment': self.risk_sentiment,
                'retail': retail_sig,
                'regime': regime_val,
            }

            # Count populated layers (non-empty values that provide actual information)
            populated = {k: v for k, v in layers.items() if v not in self.EMPTY_VALUES}
            populated_count = len(populated)

            # FIX 1: Classify each populated layer using classify_layer()
            classifications = {}
            for lname, lval in layers.items():
                if lval in self.EMPTY_VALUES:
                    classifications[lname] = 'NEUTRAL'
                else:
                    classifications[lname] = classify_layer(lname, lval, pair, direction)

            # FIX 1 & Target 4: Fractional Veto for standard retail logic
            supports = sum(1 for c in classifications.values() if c == 'SUPPORTS')
            opposes = 0.0
            for lname, c in classifications.items():
                if c == 'OPPOSES':
                    if lname == 'retail':
                        rs_tier = _rs_get(pair).get('tier', 'STANDARD')
                        if pair in InstrumentSelector.GROUP_B_PAIRS:
                            rs_tier = 'STANDARD'
                        if rs_tier == 'STANDARD':
                            opposes += 0.5
                        else:
                            opposes += 1.0
                    else:
                        opposes += 1.0

            # FIX 1: Permission logic based on supports/opposes counts
            if populated_count < 3:
                permission = 'INSUFFICIENT DATA'
                size_mod = f'— (need ≥3, have {populated_count})'
            elif opposes >= 2:
                permission = 'DO NOT TRADE'
                size_mod = '0%'
            elif opposes >= 1:
                permission = 'REDUCE SIZE'
                size_mod = '50-75%'
            elif supports >= 3 and populated_count >= 4:
                permission = 'FULL CONVICTION'
                size_mod = '100%'
            elif supports >= 2:
                permission = 'ALIGNED — LOW DATA'
                size_mod = '50-75%'
            else:
                permission = 'NEUTRAL — NO SIGNAL'
                size_mod = '—'

            # Liquidity contraction override
            if GLOBAL_LIQUIDITY in ('CONTRACTION', 'CRISIS'):
                if permission == 'FULL CONVICTION':
                    size_mod = '75% (contraction)'
                elif permission in ('REDUCE SIZE', 'ALIGNED — LOW DATA'):
                    size_mod = '25-50% (contraction)'

            matrix[pair] = {
                'layers': layers,
                'classifications': classifications,
                'direction': direction,
                'permission': permission,
                'size_modifier': size_mod,
                'populated_count': populated_count,
                'supports': supports,
                'opposes': opposes,
                'total_layers': 6,
            }
        return matrix

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ENGINE 7: INSTRUMENT SELECTION ENGINE (CHANGE 5)
# v2.3.4: Added SEVERE_RISK_OFF to REGIME_PERMISSIONS and _map_regime()
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class InstrumentSelector:
    GROUP_B_PAIRS = {
        'EUR/JPY', 'GBP/JPY', 'AUD/JPY', 'NZD/JPY', 'CAD/JPY', 'CHF/JPY',
        'EUR/GBP', 'EUR/AUD', 'EUR/CAD', 'EUR/NZD', 'EUR/CHF',
        'GBP/AUD', 'GBP/CAD', 'GBP/NZD', 'GBP/CHF',
        'AUD/NZD', 'AUD/CAD', 'AUD/CHF',
        'NZD/CAD', 'NZD/CHF', 'CAD/CHF'
    }

    # v2.3.4 (Item #2): Added SEVERE_RISK_OFF regime permissions.
    #
    # Design rationale (from analysis session):
    # - MILD_RISK_ON blocked: At VIX 30+ with backwardated curve, these fight active fear.
    #   MODERATE permits them (VIX ~21 is different); SEVERE should not. Scoring already
    #   demotes via liq_demote(-1) but blocking provides the hard gate.
    # - NEUTRAL chars permitted: AUD/NZD LONG (char NEUTRAL) was highest-conviction trade
    #   in analysis session — full macro alignment, clean range, tight spreads. Blocking
    #   NEUTRAL under SEVERE eliminates best opportunities. NEUTRAL instruments are
    #   non-directional on risk; conviction matrix evaluates them on fundamentals.
    #   EXTREME blocks NEUTRAL (VIX >40 = haven-only); at 30-40 that's overcorrection.
    # - max_instruments: 6. MODERATE=8, EXTREME=5. Midpoint 6.5 rounded down.
    # - No forced index exclusion: At VIX 30-40, index shorts (RISK_OFF char) are among
    #   best-performing trades. Force-excluding blocks both directions. Directional char
    #   system handles this: longs (RISK_ON) blocked, shorts (RISK_OFF) favored.
    #   EXTREME force-excludes because VIX >40 shorts have violent snapback risk.
    REGIME_PERMISSIONS = {
        'EXTREME_RISK_OFF': {
            'favored': ['RISK_OFF', 'ANTI_FIAT'],
            'permitted': [],
            'blocked': ['RISK_ON', 'MILD_RISK_ON', 'NEUTRAL', 'CONDITIONAL', 'MIXED'],
            'max_instruments': 5,
            'forced_includes': ['XAU/USD'],
            'forced_excludes': ['NAS100', 'SPX500', 'US30'],
        },
        'SEVERE_RISK_OFF': {
            'favored': ['RISK_OFF', 'ANTI_FIAT'],
            'permitted': ['NEUTRAL', 'MILD_RISK_OFF'],
            'blocked': ['RISK_ON', 'MILD_RISK_ON'],
            'max_instruments': 6,
            'forced_includes': ['XAU/USD'],
            'forced_excludes': [],
        },
        'MODERATE_RISK_OFF': {
            'favored': ['RISK_OFF', 'ANTI_FIAT'],
            'permitted': ['NEUTRAL', 'MILD_RISK_OFF'],
            'blocked': ['RISK_ON'],
            'max_instruments': 8,
            'forced_includes': ['XAU/USD'],
            'forced_excludes': [],
        },
        'NEUTRAL': {
            'favored': [],
            'permitted': ['RISK_ON', 'RISK_OFF', 'NEUTRAL', 'MILD_RISK_ON', 'MILD_RISK_OFF', 'CONDITIONAL', 'MIXED', 'ANTI_FIAT'],
            'blocked': [],
            'max_instruments': 10,
            'forced_includes': ['XAU/USD'],
            'forced_excludes': [],
        },
        'RISK_ON': {
            'favored': ['RISK_ON', 'MILD_RISK_ON'],
            'permitted': ['NEUTRAL', 'CONDITIONAL', 'ANTI_FIAT'],
            'blocked': [],
            'max_instruments': 10,
            'forced_includes': [],
            'forced_excludes': [],
        },
    }

    LIQUIDITY_MODIFIERS = {
        'EXPANSION': {'boost': ['RISK_ON', 'MILD_RISK_ON'], 'demote': ['RISK_OFF']},
        'LEAN_EXPANSION': {'boost': ['RISK_ON'], 'demote': []},
        'NEUTRAL': {'boost': [], 'demote': []},
        'LEAN_CONTRACTION': {'boost': ['RISK_OFF'], 'demote': ['RISK_ON']},
        'CONTRACTION': {'boost': ['RISK_OFF', 'ANTI_FIAT'], 'demote': ['RISK_ON', 'MILD_RISK_ON']},
        'CRISIS': {'boost': ['ANTI_FIAT'], 'demote': ['RISK_ON', 'MILD_RISK_ON', 'NEUTRAL']},
    }

    SESSION_COMPAT = {
        'LONDON': {'LONDON': 1, 'LONDON_NY': 1, 'ALL': 0, 'US_CASH': -2, 'NYMEX': -2},
        'NY': {'LONDON': -1, 'LONDON_NY': 0, 'US_CASH': 1, 'NYMEX': 1, 'ALL': 0},
        'OVERLAP': {'LONDON': 0, 'LONDON_NY': 1, 'US_CASH': 0, 'NYMEX': 0, 'ALL': 0},
        'ASIA': {'LONDON': -2, 'LONDON_NY': -1, 'ALL': 0, 'US_CASH': -2, 'NYMEX': -2},
    }

    CONCENTRATION_LIMITS = {
        'EUR': 2, 'GBP': 2, 'JPY': 2, 'AUD': 2, 'NZD': 2, 'CAD': 2, 'CHF': 2, 'USD': 3,
        'commodity_metals': {'RISK_ON': 2, 'NEUTRAL': 2, 'MODERATE_RISK_OFF': 1, 'SEVERE_RISK_OFF': 1, 'EXTREME_RISK_OFF': 1},
        'carry_cross': 2,
        'index': {'RISK_ON': 2, 'NEUTRAL': 2, 'MODERATE_RISK_OFF': 1, 'SEVERE_RISK_OFF': 1, 'EXTREME_RISK_OFF': 1},
    }

    MINIMUM_SCORE = 3

    def __init__(self, liquidity, risk_sentiment, fiscal, rate_diffs, retail, cot_results, session):
        self.liquidity = liquidity
        self.risk_sentiment = risk_sentiment
        self.fiscal = fiscal
        self.rate_diffs = rate_diffs
        self.retail = retail
        self.cot_results = cot_results
        self.session = session
        self._regime_key = self._map_regime(risk_sentiment)

    def _map_regime(self, rs):
        """v2.3.4 (Item #3): Added SEVERE_RISK_OFF mapping.
        Without this, SEVERE_RISK_OFF falls through to 'NEUTRAL' — the most
        permissive regime — silently bypassing all risk-off restrictions."""
        mapping = {
            'RISK_ON': 'RISK_ON',
            'NEUTRAL': 'NEUTRAL',
            'MODERATE_RISK_OFF': 'MODERATE_RISK_OFF',
            'SEVERE_RISK_OFF': 'SEVERE_RISK_OFF',
            'EXTREME_RISK_OFF': 'EXTREME_RISK_OFF',
        }
        return mapping.get(rs, 'NEUTRAL')

    def _get_currencies(self, pair):
        parts = pair.split('/')
        if len(parts) == 2:
            return [parts[0], parts[1]]
        return []

    def score_instrument(self, pair, direction):
        dna = INSTRUMENT_DNA.get(pair)
        if dna is None:
            return -100, ['NO_DNA']
        char = dna['long_char'] if direction == 'LONG' else dna['short_char']
        rp = self.REGIME_PERMISSIONS.get(self._regime_key, self.REGIME_PERMISSIONS['NEUTRAL'])
        reasons = []
        score = 0.0

        if pair in rp.get('forced_excludes', []):
            return -100, ['FORCED_EXCLUDE']
        if char in rp.get('blocked', []):
            return -100, [f'BLOCKED({char})']

        if char in rp.get('favored', []):
            score += 3
            reasons.append('regime_favored(+3)')
        elif char in rp.get('permitted', []):
            reasons.append('regime_permitted')
        else:
            return -100, [f'NOT_PERMITTED({char})']

        lm = self.LIQUIDITY_MODIFIERS.get(self.liquidity, {'boost': [], 'demote': []})
        if char in lm['boost']:
            score += 2
            reasons.append('liq_boost(+2)')
        if char in lm['demote']:
            score -= 1
            reasons.append('liq_demote(-1)')

        rd = self.rate_diffs.get(pair, {})
        if rd.get('spread_bp') is not None:
            strength = rd.get('strength', 'LOW')
            favors = rd.get('favors', '')
            rd_score = {'MAX': 3, 'STRONG': 2, 'MOD': 1, 'LOW': 0.5}.get(strength, 0)
            parts = pair.split('/')
            if len(parts) == 2:
                aligned = (direction == 'LONG' and favors == parts[0]) or (direction == 'SHORT' and favors == parts[1])
            else:
                aligned = False
            if aligned:
                score += rd_score
                reasons.append(f'rd_align(+{rd_score})')
            else:
                score -= rd_score / 2
                reasons.append(f'rd_oppose(-{rd_score/2})')

        fiscal_dir = derive_pair_fiscal(pair)
        if fiscal_dir == 'BULLISH' and direction == 'LONG':
            score += 2
            reasons.append('fiscal_align(+2)')
        elif fiscal_dir == 'BEARISH' and direction == 'SHORT':
            score += 2
            reasons.append('fiscal_align(+2)')
        elif fiscal_dir == 'BULLISH' and direction == 'SHORT':
            score -= 2
            reasons.append('fiscal_oppose(-2)')
        elif fiscal_dir == 'BEARISH' and direction == 'LONG':
            score -= 2
            reasons.append('fiscal_oppose(-2)')

        rs_data = _rs_get(pair)
        ret_sig = rs_data.get('signal', 'NO_SIGNAL')
        ret_tier = rs_data.get('tier', 'STANDARD')

        # Group B Cap override
        if pair in self.GROUP_B_PAIRS:
            ret_tier = 'STANDARD'
            
        tier_mult = {'STANDARD': 1.0, 'HIGH': 1.5, 'MAXIMUM': 2.0}.get(ret_tier, 1.0)

        if ret_sig == 'CONTRARIAN_BULL' and direction == 'LONG':
            score += (1.0 * tier_mult)
            reasons.append(f'retail(+{1.0 * tier_mult})')
        elif ret_sig == 'CONTRARIAN_BEAR' and direction == 'SHORT':
            score += (1.0 * tier_mult)
            reasons.append(f'retail(+{1.0 * tier_mult})')
        elif ret_sig == 'CONTRARIAN_BULL' and direction == 'SHORT':
            score -= (1.0 * tier_mult)
            reasons.append(f'retail(-{1.0 * tier_mult})')
        elif ret_sig == 'CONTRARIAN_BEAR' and direction == 'LONG':
            score -= (1.0 * tier_mult)
            reasons.append(f'retail(-{1.0 * tier_mult})')

        cot_map = {'EUR': '099741', 'GBP': '096742', 'JPY': '097741', 'AUD': '232741',
                   'NZD': '112741', 'CAD': '090741', 'CHF': '092741', 'XAU': '088691', 'XAG': '084691'}
        for ccy in self._get_currencies(pair):
            code = cot_map.get(ccy)
            if code and code in self.cot_results:
                cr = self.cot_results[code]
                if cr.get('setup_valid'):
                    sig = cr.get('signal', '')
                    if 'Contrarian BULL' in sig and direction == 'LONG':
                        score += 2
                        reasons.append(f'COT_{ccy}(+2)')
                    elif 'Contrarian BEAR' in sig and direction == 'SHORT':
                        score += 2
                        reasons.append(f'COT_{ccy}(+2)')

        tier = dna.get('spread_tier', 2)
        if tier == 1:
            score += 1
            reasons.append('tight_spread(+1)')
        elif tier == 3:
            score -= 1
            reasons.append('wide_spread(-1)')

        if pair in rp.get('forced_includes', []):
            score += 2
            reasons.append('forced_include(+2)')
        if dna.get('always_include'):
            score += 1
            reasons.append('always_include(+1)')

        sc = self.SESSION_COMPAT.get(self.session, {})
        sess_adj = sc.get(dna.get('session', 'ALL'), 0)
        if sess_adj != 0:
            score += sess_adj
            reasons.append(f'session({sess_adj:+d})')

        return round(score, 1), reasons

    def select_watchlist(self):
        rp = self.REGIME_PERMISSIONS.get(self._regime_key, self.REGIME_PERMISSIONS['NEUTRAL'])
        max_inst = rp.get('max_instruments', 10)

        candidates = []
        blocked_list = []
        for pair in INSTRUMENT_DNA:
            for direction in ('LONG', 'SHORT'):
                score, reasons = self.score_instrument(pair, direction)
                if score <= -100:
                    blocked_list.append({'pair': pair, 'direction': direction, 'reason': ', '.join(reasons)})
                elif score >= self.MINIMUM_SCORE:
                    candidates.append({'pair': pair, 'direction': direction, 'score': score, 'reasons': reasons})

        candidates.sort(key=lambda x: -x['score'])

        selected = []
        ccy_counts = Counter()
        cat_counts = Counter()
        seen_pairs = set()

        for c in candidates:
            pair = c['pair']
            if pair in seen_pairs:
                continue
            dna = INSTRUMENT_DNA.get(pair, {})
            currencies = self._get_currencies(pair)
            category = dna.get('category', 'other')

            blocked = False
            for ccy in currencies:
                limit = self.CONCENTRATION_LIMITS.get(ccy, 99)
                if ccy_counts[ccy] >= limit:
                    blocked = True
                    break

            if category == 'carry_cross' and cat_counts['carry_cross'] >= self.CONCENTRATION_LIMITS.get('carry_cross', 2):
                blocked = True
                
            if category == 'commodity' and pair in ('XAU/USD', 'XAG/USD'):
                cm_limits = self.CONCENTRATION_LIMITS.get('commodity_metals', {})
                cm_limit = cm_limits.get(self._regime_key, 1) if isinstance(cm_limits, dict) else 1
                if cat_counts['commodity_metals'] >= cm_limit:
                    blocked = True
                    
            if category == 'index':
                idx_limits = self.CONCENTRATION_LIMITS.get('index', {})
                idx_limit = idx_limits.get(self._regime_key, 2) if isinstance(idx_limits, dict) else 2
                if cat_counts['index'] >= idx_limit:
                    blocked = True

            if blocked:
                continue

            selected.append(c)
            seen_pairs.add(pair)
            for ccy in currencies:
                ccy_counts[ccy] += 1
            cat_counts[category] += 1
            if category == 'commodity' and pair in ('XAU/USD', 'XAG/USD'):
                cat_counts['commodity_metals'] += 1

            if len(selected) >= max_inst:
                break

        seen_blocked = set()
        unique_blocked = []
        for b in blocked_list:
            key = b['pair']
            if key not in seen_blocked:
                seen_blocked.add(key)
                unique_blocked.append(b)

        low_conviction = len(selected) < 5

        return {
            'watchlist': selected,
            'blocked': unique_blocked[:10],
            'low_conviction': low_conviction,
            'warning': f"LOW CONVICTION SESSION — only {len(selected)} instruments above threshold. Consider reduced sizing or sitting out." if low_conviction else None,
            'regime': self._regime_key,
            'max_instruments': max_inst,
        }

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RISK SENTIMENT AUTO-DERIVATION
# v2.3.4 (Item #1): Added SEVERE_RISK_OFF tier at VIX 30-40.
#
# AUTO uses VIX alone as approximation of Protocol 19's multi-factor classification.
# P19 requires: VIX level + term structure + VVIX + HY spreads + breakdown codes.
# Manual override (GLOBAL_RISK_SENTIMENT) is the precision tool.
# AUTO prevents silent NEUTRAL defaults when VIX data exists.
#
# VIX thresholds:
#   <16       → RISK_ON
#   16-20     → NEUTRAL
#   20-30     → MODERATE_RISK_OFF
#   30-40     → SEVERE_RISK_OFF  (NEW in v2.3.4)
#   >40       → EXTREME_RISK_OFF
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def derive_risk_sentiment(vix_val, manual_setting):
    """Derive risk sentiment from VIX when set to AUTO, or use manual override.

    AUTO uses VIX alone as approximation of Protocol 19's multi-factor classification.
    P19 requires: VIX level + term structure + VVIX + HY spreads + breakdown codes.
    Manual override (GLOBAL_RISK_SENTIMENT) is the precision tool.
    AUTO prevents silent NEUTRAL defaults when VIX data exists.
    """
    if manual_setting != 'AUTO':
        return manual_setting
    if vix_val is None:
        return 'NEUTRAL'
    if vix_val < 16:
        return 'RISK_ON'
    elif vix_val <= 20:
        return 'NEUTRAL'
    elif vix_val <= 30:
        return 'MODERATE_RISK_OFF'
    elif vix_val <= 40:
        return 'SEVERE_RISK_OFF'
    else:
        return 'EXTREME_RISK_OFF'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HTML DASHBOARD RENDERER (FIX 5A+5B: display truncation removed)
# v2.3.4 (Item #5): Added SEVERE RISK-OFF to regime_colors dict.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def render_dashboard(cot_results, fred_data, regime, rate_diffs,
                     conviction_matrix=None, events=None, event_warnings=None,
                     session_state=None, watchlist_data=None, risk_sentiment_used=None):
    now = datetime.now().strftime('%A %B %d, %Y — %H:%M')
    now_short = datetime.now().strftime('%Y-%m-%d')

    # v2.3.4 (Item #5): SEVERE RISK-OFF color added.
    # This is the FREDDashboard.risk_regime() display tier, not the decision-making tier.
    # FREDDashboard uses its own 5-tier VIX classification for dashboard display only.
    # The operational logic uses derive_risk_sentiment() → REGIME_PERMISSIONS → classify_layer().
    regime_colors = {
        'RISK-ON': ('#00e676', '#1b2a1b'),
        'MODERATE RISK-OFF': ('#ffab00', '#2a2517'),
        'ELEVATED RISK-OFF': ('#ff6d00', '#2a1f17'),
        'SEVERE RISK-OFF': ('#ff1744', '#2a1717'),
        'EXTREME RISK-OFF': ('#d50000', '#2a1010'),
        'UNKNOWN': ('#666', '#1a1a1a'),
    }
    r_color, r_bg = regime_colors.get(regime.get('regime', 'UNKNOWN'), ('#666', '#1a1a1a'))
    vix_val = regime.get('vix')
    vix_pct = min((vix_val or 0) / 80 * 100, 100)
    vix_color = '#00e676' if (vix_val or 0) < 20 else '#ffab00' if (vix_val or 0) < 25 else '#ff6d00' if (vix_val or 0) < 35 else '#ff1744'
    carry = regime.get('carry', 'UNKNOWN')
    carry_color = '#00e676' if carry == 'ENABLED' else '#ff1744'
    us2 = fred_data.get('DGS2', {}).get('val')
    us10 = fred_data.get('DGS10', {}).get('val')
    yc_spread, yc_text, yc_color = None, "—", "#888"
    if us2 is not None and us10 is not None:
        yc_spread = round((us10 - us2) * 100, 1)
        yc_text = f"{yc_spread:+.1f}bp"
        yc_color = '#ff5252' if yc_spread < 0 else '#69f0ae'
    hy_dir = regime.get('hy_direction', '—')
    hy_color = '#ff5252' if hy_dir == 'WIDENING' else '#69f0ae' if hy_dir == 'TIGHTENING' else '#888'

    p18_html = ""
    if regime.get('p18_watch'):
        p18_html = '<div style="background:#ff174422;border:1px solid #ff174466;border-radius:8px;padding:12px 16px;margin-bottom:20px;display:flex;align-items:center;gap:10px;"><span style="font-size:20px;">⚠️</span><div><div style="color:#ff1744;font-weight:bold;font-size:13px;">P18 SUSPENSION WATCH</div><div style="color:#ff8a80;font-size:11px;">VIX &gt; 25 — Standard trend-following protocols may need halting</div></div></div>'

    # ── SESSION STATE ──
    session_html, session_copy = "", ""
    if session_state:
        s = session_state
        positions = s.get('open_positions', [])
        pos_text_lines = ["  None"] if not positions else [f"  {p.get('instrument','—')} {p.get('direction','—')} @ {p.get('entry','—')} | Stop: {p.get('stop','—')} | {p.get('protocol','—')} | {p.get('r_status','0R')}" for p in positions]
        pos_rows = ""
        if positions:
            for p in positions:
                dc = '#69f0ae' if p.get('direction') == 'LONG' else '#ff5252'
                pos_rows += f'<tr><td style="padding:6px 12px;border-bottom:1px solid #2a2a2a;color:#e0e0e0;">{p.get("instrument","—")}</td><td style="padding:6px 12px;border-bottom:1px solid #2a2a2a;color:{dc};font-weight:bold;">{p.get("direction","—")}</td><td style="padding:6px 12px;border-bottom:1px solid #2a2a2a;font-family:monospace;color:#e0e0e0;">{p.get("entry","—")}</td><td style="padding:6px 12px;border-bottom:1px solid #2a2a2a;font-family:monospace;color:#ff5252;">{p.get("stop","—")}</td><td style="padding:6px 12px;border-bottom:1px solid #2a2a2a;color:#888;">{p.get("protocol","—")}</td><td style="padding:6px 12px;border-bottom:1px solid #2a2a2a;color:#448aff;font-weight:bold;">{p.get("r_status","0R")}</td></tr>'
        else:
            pos_rows = '<tr><td colspan="6" style="padding:12px;color:#555;text-align:center;">No open positions</td></tr>'
        spc = '#69f0ae' if s.get('session_pnl_r', 0) >= 0 else '#ff5252'
        wpc = '#69f0ae' if s.get('weekly_pnl_r', 0) >= 0 else '#ff5252'
        session_html = f'<div style="background:#141414;border:1px solid #448aff44;border-radius:8px;overflow:hidden;margin-bottom:20px;"><div style="background:#1a1a2a;padding:12px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#448aff;">🔄 Session State Recovery</span><span style="color:#666;font-size:11px;margin-left:12px;">Last: {s.get("last_updated","—")}</span></div><div style="padding:16px;"><div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px;"><div style="background:#0d0d0d;padding:12px;border-radius:6px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;">Session P&L</div><div style="font-size:20px;font-weight:bold;color:{spc};font-family:monospace;">{s.get("session_pnl_r",0):+.1f}R</div></div><div style="background:#0d0d0d;padding:12px;border-radius:6px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;">Weekly P&L</div><div style="font-size:20px;font-weight:bold;color:{wpc};font-family:monospace;">{s.get("weekly_pnl_r",0):+.1f}R</div></div><div style="background:#0d0d0d;padding:12px;border-radius:6px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;">Daily Risk</div><div style="font-size:20px;font-weight:bold;color:#e0e0e0;font-family:monospace;">{s.get("daily_risk_used_pct",0):.1f}%</div><div style="font-size:10px;color:#666;">of {s.get("daily_risk_cap_pct",3):.0f}% cap</div></div><div style="background:#0d0d0d;padding:12px;border-radius:6px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;">Liquidity</div><div style="font-size:14px;font-weight:bold;color:#e0e0e0;">{s.get("liquidity_regime","UNKNOWN")}</div></div></div><table style="width:100%;border-collapse:collapse;font-size:13px;"><tr style="color:#666;font-size:11px;text-transform:uppercase;"><th style="padding:6px 12px;text-align:left;border-bottom:1px solid #2a2a2a;">Instrument</th><th style="padding:6px 12px;text-align:left;">Dir</th><th style="padding:6px 12px;text-align:left;">Entry</th><th style="padding:6px 12px;text-align:left;">Stop</th><th style="padding:6px 12px;text-align:left;">Protocol</th><th style="padding:6px 12px;text-align:left;">R</th></tr>{pos_rows}</table></div></div>'
        session_copy = f"Open positions:\n{chr(10).join(pos_text_lines)}\nSession P&L: {s.get('session_pnl_r',0):+.1f}R\nDaily risk: {s.get('daily_risk_used_pct',0):.1f}% of {s.get('daily_risk_cap_pct',3):.0f}% cap\nWeekly P&L: {s.get('weekly_pnl_r',0):+.1f}R"

    # ── FRED TABLE ──
    fred_labels = {'VIXCLS': ('VIX', '📊'), 'BAMLH0A0HYM2': ('HY OAS', '💳'), 'DGS2': ('US 2Y', '🏛️'), 'DGS10': ('US 10Y', '🏛️'), 'DFII10': ('TIPS 10Y Real', '📈'), 'T10YIE': ('Breakeven 10Y', '🔥'), 'DTWEXBGS': ('USD Broad TWI', '💵')}
    fred_rows, fred_copy = "", []
    for sid, (label, icon) in fred_labels.items():
        d = fred_data.get(sid)
        if d:
            chg_s = f"{d['chg']:+.3f}" if d['chg'] is not None else "—"
            chg_c = '#ff5252' if d.get('chg') and d['chg'] > 0 else '#69f0ae' if d.get('chg') and d['chg'] < 0 else '#888'
            fred_rows += f'<tr><td style="padding:8px 12px;border-bottom:1px solid #2a2a2a;">{icon} {label}</td><td style="padding:8px 12px;border-bottom:1px solid #2a2a2a;text-align:right;font-family:monospace;font-weight:bold;color:#e0e0e0;">{d["val"]:.3f}</td><td style="padding:8px 12px;border-bottom:1px solid #2a2a2a;text-align:right;font-family:monospace;color:{chg_c};">{chg_s}</td><td style="padding:8px 12px;border-bottom:1px solid #2a2a2a;text-align:right;color:#666;font-size:11px;">{d["date"]}</td></tr>'
            fred_copy.append(f"  {label:<20} {d['val']:>8.3f}  chg: {chg_s}  ({d['date']})")
        else:
            fred_rows += f'<tr><td style="padding:8px 12px;border-bottom:1px solid #2a2a2a;">{icon} {label}</td><td colspan="3" style="padding:8px 12px;border-bottom:1px solid #2a2a2a;color:#555;">No data</td></tr>'
            fred_copy.append(f"  {label:<20} No data")

    # ── RATE DIFF TABLES (split majors/crosses) ──
    def build_rate_row(pair, d):
        if d.get('spread_bp') is None:
            return f'<tr><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;font-weight:bold;">{pair}</td><td colspan="4" style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:#555;">{d.get("note","—")}</td></tr>', f"  {pair:<10} MISSING"
        sp = d['spread_bp']
        bar_c = '#448aff' if 'USD' in d.get('favors', '') else '#ffab00'
        str_c = {'MAX': '#ff1744', 'STRONG': '#ff6d00', 'MOD': '#ffab00', 'LOW': '#66bb6a'}.get(d['strength'], '#888')
        shift_h = f'<span style="font-family:monospace;color:{"#ff1744" if d.get("shift_class")=="STRONG" else "#ffab00" if d.get("shift_class")=="MODERATE" else "#666"};font-size:11px;">{d["shift_bp"]:+.1f}bp</span>' if d.get('shift_bp') is not None else '—'
        stale_b = '<span style="color:#ff1744;font-size:9px;margin-left:4px;">STALE</span>' if d.get('stale') else ''
        row = f'<tr><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;font-weight:bold;color:#e0e0e0;font-size:12px;">{pair}{stale_b}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;text-align:right;font-family:monospace;color:{bar_c};font-size:12px;font-weight:bold;">{sp:+.1f}bp→{d["favors"]}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;text-align:center;"><span style="background:{str_c};color:#000;padding:1px 6px;border-radius:3px;font-size:10px;font-weight:bold;">{d["strength"]}</span></td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;text-align:right;">{shift_h}</td></tr>'
        stale_t = " ⚠STALE" if d.get('stale') else ""
        shift_t = f" shift:{d['shift_bp']:+.1f}bp" if d.get('shift_bp') is not None else ""
        copy = f"  {pair:<10} {sp:>+7.1f}bp → {d['favors']:<4} [{d['strength']}]{shift_t}{stale_t}"
        return row, copy

    major_rows, major_copy, cross_rows, cross_copy = "", [], "", []
    for pair, d in rate_diffs.items():
        row, copy = build_rate_row(pair, d)
        if d.get('pair_type') == 'major':
            major_rows += row
            major_copy.append(copy)
        else:
            cross_rows += row
            cross_copy.append(copy)

    rate_html = f"""
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:20px;">
        <div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;">
            <div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#e0e0e0;">💱 Majors — Rate Differentials vs USD</span></div>
            <table style="width:100%;border-collapse:collapse;font-size:12px;">
                <tr style="color:#666;font-size:10px;text-transform:uppercase;"><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Pair</th><th style="padding:6px 10px;text-align:right;border-bottom:1px solid #2a2a2a;">Spread</th><th style="padding:6px 10px;text-align:center;border-bottom:1px solid #2a2a2a;">Str</th><th style="padding:6px 10px;text-align:right;border-bottom:1px solid #2a2a2a;">Shift</th></tr>
                {major_rows}
            </table>
        </div>
        <div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;">
            <div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#e0e0e0;">💱 Crosses — Rate Differentials (derived)</span></div>
            <div style="max-height:400px;overflow-y:auto;">
            <table style="width:100%;border-collapse:collapse;font-size:12px;">
                <tr style="color:#666;font-size:10px;text-transform:uppercase;"><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Pair</th><th style="padding:6px 10px;text-align:right;border-bottom:1px solid #2a2a2a;">Spread</th><th style="padding:6px 10px;text-align:center;border-bottom:1px solid #2a2a2a;">Str</th><th style="padding:6px 10px;text-align:right;border-bottom:1px solid #2a2a2a;">Shift</th></tr>
                {cross_rows}
            </table>
            </div>
        </div>
    </div>"""

    # ── COT CARDS (compact) ──
    # FIX 5B: Removed sig[:60] truncation in HTML card and sig[:50] in copy block.
    # Full signal string now displayed in both locations so %-L/%-S confirmation
    # tags are always visible. Max signal length is ~66-70 chars.
    cot_cards, cot_copy = "", []
    for code, r in cot_results.items():
        if r.get('willco') is None:
            cot_cards += f'<div style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:6px;padding:10px;min-width:200px;"><div style="font-weight:bold;color:#888;font-size:12px;">{r["instrument"]}</div><div style="color:#555;font-size:11px;">{r.get("error","No data")}</div></div>'
            cot_copy.append(f"  {r['instrument']}: NO DATA")
            continue
        wc = r['willco']
        wc_c = '#ff1744' if wc >= 80 else '#ff6d00' if wc >= 75 else '#00e676' if wc <= 20 else '#69f0ae' if wc <= 25 else '#448aff'
        sig = r['signal']
        sig_c = '#ff1744' if 'EXTREME LONG' in sig else '#00e676' if 'EXTREME SHORT' in sig else '#ff6d00' if 'ELEVATED' in sig else '#69f0ae' if 'DEPRESSED' in sig else '#666'
        rpt = r.get('report_used', '—')
        rpt_short = 'Legacy⚠' if 'Legacy' in str(rpt) and r.get('is_fallback') else 'Legacy' if 'Legacy' in str(rpt) else 'MM' if 'Managed' in str(rpt) else 'LF' if 'Leveraged' in str(rpt) else '?'
        rpt_c = '#ff5252' if r.get('is_fallback') else '#69f0ae'
        gate_c = '#00e676' if r['oi_gate'] else '#ff5252'
        # FIX 5B: sig displayed in full (no [:60] truncation)
        cot_cards += f"""<div style="background:#1a1a1a;border:1px solid {'#00e67644' if r.get('setup_valid') else '#2a2a2a'};border-radius:6px;padding:12px;min-width:220px;flex:1;">
            <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;"><span style="font-weight:bold;color:#e0e0e0;font-size:13px;">{r['instrument']}</span><span style="font-size:9px;color:{rpt_c};background:{rpt_c}15;padding:1px 6px;border-radius:3px;">{rpt_short}</span></div>
            <div style="text-align:center;"><div style="font-size:30px;font-weight:bold;color:{wc_c};font-family:monospace;">{wc:.1f}</div><div style="font-size:9px;color:#666;">WillCo ({r['window']}w)</div></div>
            <div style="background:{sig_c}15;border:1px solid {sig_c}33;border-radius:3px;padding:4px 8px;text-align:center;font-size:10px;color:{sig_c};margin:6px 0;font-weight:bold;">{sig}</div>
            <div style="display:grid;grid-template-columns:1fr 1fr;gap:2px;font-size:11px;">
                <div style="color:#888;">Net</div><div style="text-align:right;font-family:monospace;color:#e0e0e0;">{r['net']:+,}</div>
                <div style="color:#888;">1w</div><div style="text-align:right;font-family:monospace;color:{'#69f0ae' if r['net_chg_1w']>0 else '#ff5252' if r['net_chg_1w']<0 else '#888'};">{r['net_chg_1w']:+,}</div>
                <div style="color:#888;">OI Gate</div><div style="text-align:right;color:{gate_c};font-weight:bold;">{'PASS' if r['oi_gate'] else 'FAIL'}</div>
                <div style="color:#888;">As Of</div><div style="text-align:right;color:#666;font-size:10px;">{r['as_of']}</div>
            </div></div>"""
        # FIX 5B: sig displayed in full (no [:50] truncation)
        cot_copy.append(f"  {r['instrument']:<20} WillCo {wc:>5.1f} | Net:{r['net']:+,} | OI:{'PASS' if r['oi_gate'] else 'FAIL'} | [{rpt_short}] | {sig}")

    # ── WATCHLIST TABLE ──
    # FIX 5A: Removed [:4] slice on reasons. All scoring components now shown in
    # both the HTML table and the copy block so displayed breakdown sums to total score.
    watchlist_html, watchlist_copy = "", []
    if watchlist_data:
        wl = watchlist_data
        wl_rows = ""
        for i, item in enumerate(wl.get('watchlist', []), 1):
            sc = item['score']
            sc_c = '#00e676' if sc >= 7 else '#69f0ae' if sc >= 5 else '#ffab00'
            dir_c = '#69f0ae' if item['direction'] == 'LONG' else '#ff5252'
            # FIX 5A: Show ALL reasons (no [:4] truncation)
            reasons_str = ', '.join(item['reasons'])
            wl_rows += f'<tr><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:#888;font-weight:bold;">{i}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:#e0e0e0;font-weight:bold;">{item["pair"]}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:{dir_c};font-weight:bold;">{item["direction"]}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;text-align:right;font-family:monospace;color:{sc_c};font-weight:bold;">{sc:+.1f}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:#888;font-size:10px;">{reasons_str}</td></tr>'
            # FIX 5A: Copy block also shows all reasons
            watchlist_copy.append(f"  {i}. {item['pair']} {item['direction']} [Score {sc:+.1f}] — {reasons_str}")

        warn_html = ""
        if wl.get('warning'):
            warn_html = f'<div style="background:#ffab0022;border:1px solid #ffab0044;border-radius:6px;padding:8px 12px;margin-bottom:12px;color:#ffab00;font-size:12px;font-weight:bold;">⚠ {wl["warning"]}</div>'

        blocked_html = ""
        if wl.get('blocked'):
            bl_items = ', '.join(f"{b['pair']}" for b in wl['blocked'][:8])
            blocked_html = f'<div style="margin-top:8px;font-size:11px;color:#555;">Blocked: {bl_items}</div>'

        watchlist_html = f"""
        <div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;margin-bottom:20px;">
            <div style="background:#1a1a1a;padding:12px 16px;border-bottom:1px solid #2a2a2a;">
                <span style="font-weight:bold;color:#e0e0e0;">🎯 Instrument Selection Watchlist</span>
                <span style="color:#666;font-size:11px;margin-left:8px;">Regime: {wl.get('regime','?')} | Max: {wl.get('max_instruments','?')}</span>
            </div>
            <div style="padding:12px 16px;">
                {warn_html}
                <table style="width:100%;border-collapse:collapse;font-size:13px;">
                    <tr style="color:#666;font-size:10px;text-transform:uppercase;"><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">#</th><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Instrument</th><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Dir</th><th style="padding:6px 10px;text-align:right;border-bottom:1px solid #2a2a2a;">Score</th><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Reasons</th></tr>
                    {wl_rows}
                </table>
                {blocked_html}
                <div style="margin-top:10px;font-size:11px;color:#448aff;">Next: Open TradingView, check P16 regime on these {len(wl.get('watchlist',[]))} charts</div>
            </div>
        </div>"""

    # ── CONVICTION MATRIX (FIX 1: now shows S/O/N + raw value + direction) ──
    conviction_html, conviction_copy = "", []
    if conviction_matrix:
        cm_rows = ""
        for pair, cm in conviction_matrix.items():
            layers = cm['layers']
            cls = cm.get('classifications', {})
            direction = cm.get('direction', '?')
            perm = cm['permission']
            sz = cm['size_modifier']
            pop = cm.get('populated_count', 0)
            sup = cm.get('supports', 0)
            opp = cm.get('opposes', 0)
            perm_c = '#00e676' if perm == 'FULL CONVICTION' else '#69f0ae' if 'ALIGNED' in perm else '#ffab00' if perm == 'REDUCE SIZE' else '#ff1744' if perm == 'DO NOT TRADE' else '#666'
            dir_c = '#69f0ae' if direction == 'LONG' else '#ff5252' if direction == 'SHORT' else '#888'

            def lcell(layer_name, raw_val):
                """Render a cell showing normalized S/O/N with raw value as tooltip."""
                c = cls.get(layer_name, 'NEUTRAL')
                if raw_val in ('—', 'UNKNOWN', '', 'NOT_ASSESSED', 'NO_SIGNAL'):
                    return f'<td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;color:#555;font-size:10px;" title="{raw_val}">—</td>'
                elif c == 'SUPPORTS':
                    return f'<td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;color:#69f0ae;font-size:10px;font-weight:bold;" title="{raw_val}">✓ S</td>'
                elif c == 'OPPOSES':
                    return f'<td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;color:#ff5252;font-size:10px;font-weight:bold;" title="{raw_val}">✗ O</td>'
                else:
                    return f'<td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;color:#888;font-size:10px;" title="{raw_val}">— N</td>'

            lc_c = '#00e676' if pop >= 4 else '#ffab00' if pop >= 3 else '#ff5252'
            cm_rows += f'<tr><td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;font-weight:bold;color:#e0e0e0;font-size:11px;">{pair}</td><td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;color:{dir_c};font-weight:bold;font-size:10px;">{direction}</td>{lcell("liquidity", layers.get("liquidity","—"))}{lcell("fiscal", layers.get("fiscal","—"))}{lcell("rate_diff", layers.get("rate_diff","—"))}{lcell("risk_sentiment", layers.get("risk_sentiment","—"))}{lcell("retail", layers.get("retail","—"))}{lcell("regime", layers.get("regime","—"))}<td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;color:{perm_c};font-weight:bold;font-size:10px;">{perm}</td><td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;font-family:monospace;font-size:10px;">{sz}</td><td style="padding:5px 6px;border-bottom:1px solid #2a2a2a;text-align:center;color:{lc_c};font-size:10px;">{sup}S/{opp}O</td></tr>'
            conviction_copy.append(f"  {pair:<10} {direction:<5} [{sup}S/{opp}O] [{pop}/6] → {perm} ({sz})")

        conviction_html = f'<div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;margin-bottom:20px;"><div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#e0e0e0;">🎯 Conviction Matrix</span><span style="color:#666;font-size:10px;margin-left:12px;">✓S = Supports trade | ✗O = Opposes | —N = Neutral | Hover for raw value</span></div><div style="overflow-x:auto;"><table style="width:100%;border-collapse:collapse;font-size:12px;"><tr style="color:#666;font-size:9px;text-transform:uppercase;"><th style="padding:6px;text-align:left;border-bottom:1px solid #2a2a2a;">Pair</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Dir</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Liq</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Fiscal</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">RateDiff</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Risk</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Retail</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Regime</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Permission</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Size</th><th style="padding:6px;text-align:center;border-bottom:1px solid #2a2a2a;">Score</th></tr>{cm_rows}</table></div></div>'

    # ── EVENT CALENDAR ──
    events_html, events_copy = "", []
    ev_warnings_html = ""
    if event_warnings:
        for w in event_warnings:
            ev_warnings_html += f'<div style="background:#ff174422;border:1px solid #ff174444;border-radius:6px;padding:8px 12px;margin-bottom:8px;color:#ff8a80;font-size:12px;font-weight:bold;">{w["text"]}</div>'

    if events:
        ev_rows = ""
        active_bo = set()
        for ev in events:
            status = ev.get('status', 'UNKNOWN')
            stc = '#ff1744' if status == 'BLACKOUT' else '#ffab00' if status == 'UPCOMING' else '#555'
            stbg = '#ff174422' if status == 'BLACKOUT' else '#ffab0011' if status == 'UPCOMING' else 'transparent'
            tc_map = {1: '#ff1744', 2: '#ffab00', 3: '#666'}
            tc = tc_map.get(ev.get('tier', 2), '#666')
            src = ev.get('source', 'MANUAL')
            src_badge = f'<span style="font-size:8px;color:#448aff;margin-left:4px;">[FF]</span>' if src == 'FF_API' else ''
            # FIX 3: Show affected instruments for non-ALL events
            inst_list = ev.get('instruments', [])
            if inst_list and inst_list != ['ALL']:
                inst_badge = f'<span style="font-size:8px;color:#888;margin-left:4px;">({", ".join(inst_list[:4])}{"..." if len(inst_list) > 4 else ""})</span>'
            elif inst_list == ['ALL']:
                inst_badge = '<span style="font-size:8px;color:#888;margin-left:4px;">(ALL)</span>'
            else:
                inst_badge = ''
            if status == 'BLACKOUT':
                active_bo.update(ev.get('instruments', []))
            ev_rows += f'<tr style="background:{stbg};"><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:{stc};font-weight:bold;font-size:12px;">{ev.get("datetime_str","—")}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:#e0e0e0;font-size:12px;">{ev.get("event","—")}{src_badge}{inst_badge}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;text-align:center;color:{tc};font-weight:bold;">T{ev.get("tier","?")}</td><td style="padding:6px 10px;border-bottom:1px solid #2a2a2a;color:{stc};font-size:11px;">{ev.get("countdown_str","—")}</td></tr>'
            events_copy.append(f"  {ev.get('datetime_str','—'):<20} {ev.get('event','—'):<25} T{ev.get('tier','?')} | {', '.join(inst_list[:3])} | {ev.get('countdown_str','—')}")
        bo_banner = f'<div style="background:#ff174433;border:1px solid #ff174466;border-radius:6px;padding:8px 12px;margin-bottom:8px;"><span style="color:#ff1744;font-weight:bold;">🚫 ACTIVE BLACKOUT:</span> <span style="color:#ff8a80;">{", ".join(sorted(active_bo))}</span></div>' if active_bo else ''
        events_html = f'<div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;margin-bottom:20px;"><div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#e0e0e0;">📅 Events & Blackouts</span></div><div style="padding:12px 16px;">{ev_warnings_html}{bo_banner}<table style="width:100%;border-collapse:collapse;font-size:12px;"><tr style="color:#666;font-size:10px;text-transform:uppercase;"><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Date/Time</th><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Event</th><th style="padding:6px 10px;text-align:center;border-bottom:1px solid #2a2a2a;">Tier</th><th style="padding:6px 10px;text-align:left;border-bottom:1px solid #2a2a2a;">Status</th></tr>{ev_rows}</table></div></div>'
    else:
        events_html = '<div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;margin-bottom:20px;"><div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#e0e0e0;">📅 Events & Blackouts</span></div><div style="padding:16px;color:#555;text-align:center;font-size:13px;">No events this week (FF API + manual).</div></div>'
        events_copy.append("  No events.")

    # ── DATA GAPS ──
    gaps = [("Weekly ADX per instrument", "TradingView ≤ 25 for P22"), ("VIX term structure", "vixcentral.com"), ("Oil futures curve", "Contango/backwardation"), ("P16 regimes", "TradingView"), ("Retail sentiment", "FXSSI (primary) + Myfxbook (verification)")]
    for ccy, info in FOREIGN_YIELDS.items():
        if isinstance(info, dict):
            try:
                days_old = (datetime.now() - datetime.strptime(info.get('updated', '2000-01-01'), '%Y-%m-%d')).days
                if days_old > 7:
                    gaps.insert(0, (f"{ccy} 2Y yield (STALE — {days_old}d)", f"Updated: {info['updated']}"))
            except Exception:
                pass
    gap_rows, gap_copy = "", []
    for item, note in gaps:
        ic = '#ff1744' if 'STALE' in item else '#ffab00'
        gap_rows += f'<tr><td style="padding:5px 12px;border-bottom:1px solid #2a2a2a;color:{ic};font-size:12px;">⬜ {item}</td><td style="padding:5px 12px;border-bottom:1px solid #2a2a2a;color:#888;font-size:11px;">{note}</td></tr>'
        gap_copy.append(f"  [ ] {item} — {note}")

    # ── COPY BLOCK ──
    copy_block = f"""SESSION STATE BRIEF — {now_short}
{'='*60}

{session_copy if session_copy else 'Open positions: None' + chr(10) + 'Session P&L: 0R' + chr(10) + 'Daily risk: 0% of 3% cap' + chr(10) + 'Weekly P&L: 0R'}

MACRO BIAS (assessed {now_short}):
  Session: {CURRENT_SESSION} | Liquidity: {GLOBAL_LIQUIDITY} | Risk: {risk_sentiment_used or 'UNKNOWN'}
  VIX: {vix_val if vix_val else '—'} | Carry: {carry} | HY: {hy_dir or '—'} | YC 2s10s: {yc_text}

FRED DATA:
{chr(10).join(fred_copy) if fred_copy else '  No data'}

RATE DIFFERENTIALS — MAJORS (P18):
{chr(10).join(major_copy) if major_copy else '  No data'}

RATE DIFFERENTIALS — CROSSES:
{chr(10).join(cross_copy) if cross_copy else '  No data'}

COT SIGNALS (P22):
{chr(10).join(cot_copy) if cot_copy else '  No data'}

WATCHLIST ({len(watchlist_data.get('watchlist',[])) if watchlist_data else 0} instruments):
{chr(10).join(watchlist_copy) if watchlist_copy else '  Not generated'}

CONVICTION MATRIX:
{chr(10).join(conviction_copy) if conviction_copy else '  Not generated'}

UPCOMING EVENTS:
{chr(10).join(events_copy)}

MANUAL GAPS:
{chr(10).join(gap_copy)}
"""
    copy_esc = copy_block.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')

    # ── ASSEMBLE HTML ──
    html = f"""
    <div style="background:#0d0d0d;color:#ccc;font-family:'Segoe UI',system-ui,sans-serif;padding:24px;border-radius:12px;max-width:1400px;margin:0 auto;">
        <div style="border-bottom:2px solid #1a1a1a;padding-bottom:16px;margin-bottom:20px;">
            <div style="display:flex;justify-content:space-between;align-items:center;">
                <div>
                    <div style="font-size:24px;font-weight:bold;color:#e0e0e0;letter-spacing:1px;">⚡ PROTOCOL ENGINE <span style="color:#448aff;">v2.4.4</span></div>
                    <div style="font-size:12px;color:#666;margin-top:4px;">Session: {CURRENT_SESSION} | Liquidity: {GLOBAL_LIQUIDITY} | Risk: {risk_sentiment_used or 'AUTO'} | 28-pair rate diffs | FF events (country-aware) | Normalized conviction | SEVERE tier</div>
                </div>
                <div style="text-align:right;"><div style="font-size:11px;color:#888;">{now}</div></div>
            </div>
        </div>
        {p18_html}{session_html}
        <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:20px;">
            <div style="background:{r_bg};border:1px solid {r_color}44;border-radius:8px;padding:16px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;margin-bottom:6px;">Risk Regime</div><div style="font-size:18px;font-weight:bold;color:{r_color};">{regime.get('regime','UNKNOWN')}</div></div>
            <div style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:8px;padding:16px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;margin-bottom:6px;">📊 VIX</div><div style="font-size:28px;font-weight:bold;color:{vix_color};font-family:monospace;">{vix_val if vix_val else '—'}</div><div style="margin:8px auto;background:#0d0d0d;border-radius:3px;height:6px;width:100%;"><div style="background:{vix_color};height:6px;border-radius:3px;width:{vix_pct}%;"></div></div></div>
            <div style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:8px;padding:16px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;margin-bottom:6px;">Carry</div><div style="font-size:18px;font-weight:bold;color:{carry_color};">{carry}</div><div style="font-size:11px;color:{hy_color};margin-top:4px;">HY: {hy_dir or '—'}</div></div>
            <div style="background:#1a1a1a;border:1px solid #2a2a2a;border-radius:8px;padding:16px;text-align:center;"><div style="font-size:10px;color:#888;text-transform:uppercase;margin-bottom:6px;">🏛️ YC 2s10s</div><div style="font-size:24px;font-weight:bold;color:{yc_color};font-family:monospace;">{yc_text}</div></div>
        </div>
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:20px;">
            <div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;">
                <div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#e0e0e0;">📡 FRED Macro</span></div>
                <table style="width:100%;border-collapse:collapse;font-size:12px;"><tr style="color:#666;font-size:10px;text-transform:uppercase;"><th style="padding:6px 12px;text-align:left;border-bottom:1px solid #2a2a2a;">Series</th><th style="padding:6px 12px;text-align:right;border-bottom:1px solid #2a2a2a;">Value</th><th style="padding:6px 12px;text-align:right;border-bottom:1px solid #2a2a2a;">Chg</th><th style="padding:6px 12px;text-align:right;border-bottom:1px solid #2a2a2a;">Date</th></tr>{fred_rows}</table>
            </div>
            <div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;">
                <div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#e0e0e0;">📋 P22 COT WillCo</span></div>
                <div style="padding:12px;display:flex;flex-wrap:wrap;gap:8px;">{cot_cards}</div>
            </div>
        </div>
        {rate_html}
        {watchlist_html}
        {conviction_html}
        {events_html}
        <div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;margin-bottom:20px;"><div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#ffab00;">⚠️ Manual Data Gaps</span></div><table style="width:100%;border-collapse:collapse;">{gap_rows}</table></div>
        <div style="background:#141414;border:1px solid #2a2a2a;border-radius:8px;overflow:hidden;margin-bottom:20px;"><div style="background:#1a1a1a;padding:10px 16px;border-bottom:1px solid #2a2a2a;"><span style="font-weight:bold;color:#448aff;">📋 Copy Block</span></div><pre style="background:#0a0a0a;color:#b0b0b0;padding:16px;margin:0;font-size:11px;font-family:Consolas,monospace;white-space:pre-wrap;max-height:500px;overflow-y:auto;">{copy_esc}</pre></div>
        <div style="text-align:center;padding:12px;color:#444;font-size:11px;border-top:1px solid #1a1a1a;">Protocol Engine v2.4.4 — 28-pair rate diffs | FF events (country-aware) | Instrument selection | Normalized conviction | SEVERE_RISK_OFF tier</div>

# FISCAL_BIAS staleness check
from datetime import datetime as _dt_check
for _fb_ccy, _fb_entry in FISCAL_BIAS.items():
    if isinstance(_fb_entry, dict) and _fb_entry.get('updated'):
        try:
            _fb_age = (_dt_check.now() - _dt_check.strptime(_fb_entry['updated'], '%Y-%m-%d')).days
            if _fb_age > 30:
                print(f"  ⚠ [WARNING: FISCAL_BIAS {_fb_ccy} is stale ({_fb_age} days old, >30 day limit). Manual update required.]")
        except Exception:
            pass

# RETAIL_SENTIMENT staleness check
for _rs_pair, _rs_entry in RETAIL_SENTIMENT.items():
    _rs_updated = _rs_entry.get('updated')
    if not _rs_updated:
        print(f"  ⚠ [WARNING: RETAIL_SENTIMENT {_rs_pair} is missing an update date. Manual update required.]")
    else:
        try:
            _rs_age = (_dt_check.now() - _dt_check.strptime(_rs_updated, '%Y-%m-%d')).days
            if _rs_age > 1:
                print(f"  ⚠ [WARNING: RETAIL_SENTIMENT {_rs_pair} is stale ({_rs_age} days old, >1 day limit). Manual update required.]")
        except Exception:
            print(f"  ⚠ [WARNING: RETAIL_SENTIMENT {_rs_pair} has invalid date format. Manual update required.]")
    </div>"""
    display(HTML(html))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MAIN EXECUTION (FIX 1: updated Step 5 to pass direction tuples)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def run_engine(quick=False):
    mode = "QUICK RUN (cached COT)" if quick else "FULL RUN (downloading COT)"
    print("=" * 60)
    print(f" PROTOCOL ENGINE v2.4.4 — {mode}")
    print(f" {datetime.now().strftime('%A %B %d, %Y %H:%M')}")
    print(f" Session: {CURRENT_SESSION} | Liquidity: {GLOBAL_LIQUIDITY}")
    print("=" * 60)

    # Step 0: Session State
    print("\n>> STEP 0: Session State")
    session = SessionState(SESSION_DATA_DIR)
    session_state = session.state

    # Step 1: COT
    print(f"\n>> STEP 1: P22 COT ({'cached' if quick else 'download'})")
    cot = COTProcessor(COT_DATA_DIR)
    if quick:
        cot.load_cached()
    else:
        cot.download()
    cot_results = {}
    if any(df is not None for df in cot.raw.values()):
        cot_results = cot.all_results()
        print("\n  REPORT TYPE SELECTION:")
        for code, r in cot_results.items():
            wc_str = f"WillCo {r['willco']:.1f}" if r.get('willco') is not None else "NO DATA"
            fb = " ⚠FB" if r.get('is_fallback') else ""
            print(f"  {r['instrument']:<20} → {r.get('report_used','N/A')}{fb} | {wc_str}")

    # Step 2: FRED
    fred_data = {}
    regime = {'regime': 'UNKNOWN', 'vix': None, 'carry': 'UNKNOWN', 'p18_watch': False, 'p18_suspended': False, 'hy_direction': None}
    rate_diffs = {}
    if FRED_KEY:
        print("\n>> STEP 2: FRED Macro")
        fred = FREDDashboard(FRED_KEY)
        fred.fetch_all()
        fred_data = fred.data
        regime = fred.risk_regime(session_state=session)
        print(f"  Regime: {regime['regime']} | Carry: {regime['carry']}")

        us2y = fred_data.get('DGS2', {}).get('val')
        if us2y:
            print("\n>> STEP 3: P18 Rate Differentials (28 pairs)")
            rd = RateDiffEngine(us2y, FOREIGN_YIELDS, SESSION_DATA_DIR)
            rate_diffs = rd.spreads()
            majors_shown = [p for p, d in rate_diffs.items() if d.get('pair_type') == 'major' and d.get('spread_bp') is not None]
            crosses_shown = [p for p, d in rate_diffs.items() if d.get('pair_type') == 'cross' and d.get('spread_bp') is not None]
            print(f"  {len(majors_shown)} majors, {len(crosses_shown)} crosses computed")

    # Derive risk sentiment
    vix_val = fred_data.get('VIXCLS', {}).get('val')
    risk_sentiment_used = derive_risk_sentiment(vix_val, GLOBAL_RISK_SENTIMENT)
    print(f"\n  Risk Sentiment (derived): {risk_sentiment_used}")

    # Step 4: Events
    print("\n>> STEP 4: Event Calendar (FF API + manual) — country-aware matching")
    event_cal = EventCalendar(UPCOMING_EVENTS)
    events_list = event_cal.get_events()
    event_warnings = event_cal.get_warnings()
    blackouts = event_cal.active_blackouts()
    upcoming = [e for e in events_list if e.get('status') in ('UPCOMING', 'BLACKOUT')]
    print(f"  {len(events_list)} total events, {len(upcoming)} upcoming/active")
    for w in event_warnings:
        print(f"  {w['text']}")
    if blackouts:
        print(f"  🚫 ACTIVE BLACKOUTS: {', '.join(sorted(blackouts))}")

    # Step 4.5: Instrument Selection
    print("\n>> STEP 4.5: Instrument Selection")
    selector = InstrumentSelector(
        liquidity=GLOBAL_LIQUIDITY, risk_sentiment=risk_sentiment_used,
        fiscal=FISCAL_BIAS, rate_diffs=rate_diffs, retail=RETAIL_SENTIMENT,
        cot_results=cot_results, session=CURRENT_SESSION)
    watchlist_data = selector.select_watchlist()
    wl = watchlist_data.get('watchlist', [])
    print(f"  Selected {len(wl)} instruments (regime: {watchlist_data.get('regime')})")
    for i, item in enumerate(wl, 1):
        print(f"  {i}. {item['pair']} {item['direction']} [Score {item['score']:+.1f}]")
    if watchlist_data.get('warning'):
        print(f"  ⚠ {watchlist_data['warning']}")

    # Step 5: Conviction Matrix — FIX 1: pass (pair, direction) tuples
    print("\n>> STEP 5: Conviction Matrix")
    selected_instruments = [(item['pair'], item['direction']) for item in wl]
    cm_engine = ConvictionMatrix(rate_diffs=rate_diffs, risk_sentiment=risk_sentiment_used,
                                 selected_instruments=selected_instruments)
    conviction_matrix = cm_engine.build()
    for pair, cm in conviction_matrix.items():
        sup = cm.get('supports', 0)
        opp = cm.get('opposes', 0)
        print(f"  {pair:<10} {cm.get('direction','?'):<5} → {cm['permission']} ({cm['size_modifier']}) [{sup}S/{opp}O] [{cm.get('populated_count',0)}/6]")

    # Step 6: Save
    print("\n>> STEP 6: Saving session state")
    session.update(macro_assessment_date=datetime.now().strftime('%Y-%m-%d'), liquidity_regime=GLOBAL_LIQUIDITY)
    session.save()

    # Render
    print("\n>> Rendering dashboard...")
    render_dashboard(
        cot_results=cot_results, fred_data=fred_data, regime=regime, rate_diffs=rate_diffs,
        conviction_matrix=conviction_matrix, events=events_list, event_warnings=event_warnings,
        session_state=session_state, watchlist_data=watchlist_data, risk_sentiment_used=risk_sentiment_used)
    print(f"✅ Done — Protocol Engine v2.4.4 ({mode})")

# ── RUN ──
run_engine(quick=False)

In [ ]:
# CELL 2: PER-SESSION CONFIG + QUICK RUN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Fill values below, then Shift+Enter to re-run with updates.
# Only edit what changed — everything else carries from Cell 1.

# ── SESSION ──
CURRENT_SESSION = 'OVERLAP'   # LONDON | NY | OVERLAP | ASIA

# ── RISK SENTIMENT (after checking vixcentral + VVIX) ──
# VIX: ___  | Curve: contango / backwardation | VVIX: ___  | HY: widening / tightening
GLOBAL_RISK_SENTIMENT = 'MODERATE_RISK_OFF'
# Options: AUTO | RISK_ON | NEUTRAL | MODERATE_RISK_OFF | EXTREME_RISK_OFF

# ── P16 REGIME — TradingView ADX(14) for each watchlist instrument ──
# Values: TRENDING | RANGING | TRANSITIONAL
# NAS100 ADX threshold: standard ≥25
INSTRUMENT_REGIME = {
    'XAU/USD': {'regime': 'TRANSITIONAL', 'adx': 30.01},
    'AUD/NZD': {'regime': 'RANGING', 'adx': 21.02},
    'GBP/USD': {'regime': 'RANGING', 'adx': 15.73},
    'USD/CAD': {'regime': 'TRANSITIONAL', 'adx': 32.65},
    'AUD/CAD': {'regime': 'RANGING', 'adx': 14.43},
    'GBP/CHF': {'regime': 'TRANSITIONAL', 'adx': 34.06},
    'NZD/CHF': {'regime': 'TRANSITIONAL', 'adx': 29.79},
    'NAS100':  {'regime': 'RANGING', 'adx': 24.75},
}

# ── P22 WEEKLY ADX — TradingView weekly chart, for COT currencies ──
# Only needed for currencies with active/near-active COT signals
# Currently relevant: EUR (WillCo 0), NZD (WillCo 0), WTI (WillCo 100)
COT_WEEKLY_ADX = {
    'EUR': 16.71,   # Extreme SHORT — needs ADX ≤25 for P22 entry
    'NZD': 19.76,   # Extreme SHORT — needs ADX ≤25 for P22 entry
    'WTI': 34.93,   # Extreme LONG  — needs ADX ≤25 for P22 entry
}

# ── RETAIL SENTIMENT — update only changed instruments ──
# After checking FXSSI + Myfxbook this session
# Uncomment and update any that changed:
# RETAIL_SENTIMENT['NAS100']  = 'NO_SIGNAL'      # or CONTRARIAN_BULL / CONTRARIAN_BEAR
# RETAIL_SENTIMENT['SPX500']  = 'NO_SIGNAL'
# RETAIL_SENTIMENT['US30']    = 'NO_SIGNAL'
# RETAIL_SENTIMENT['WTI']     = 'CONTRARIAN_BEAR' # verify current reading
# RETAIL_SENTIMENT['XAU/USD'] = 'NO_SIGNAL'
# RETAIL_SENTIMENT['AUD/NZD'] = 'CONTRARIAN_BULL' # verify current reading
# RETAIL_SENTIMENT['GBP/USD'] = 'NO_SIGNAL'
# RETAIL_SENTIMENT['USD/CAD'] = 'CONTRARIAN_BULL' # verify current reading

# ── RUN ──
run_engine(quick=True)
